In [1]:
%load_ext cudf.pandas

In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

Enabled rmm statistics


# *Step 1: Import Python libraries*

In [3]:
%%time
### cell 0 ###

import os
import glob
import numpy as np
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
import warnings
import time

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 5000)

CPU times: user 119 ms, sys: 7.9 ms, total: 127 ms
Wall time: 125 ms


In [4]:
start = time.time()

In [5]:
%%time
### cell 1 ###


def load_survey_data(base_dir, file_name, rows_to_skip=1):
    file_path = os.path.join(base_dir, file_name)
    df = pd.read_csv(
        file_path, low_memory=False, encoding="ISO-8859-1", skiprows=rows_to_skip
    )
    file_name = "kaggle_survey_" + base_dir[-5:-1] + ".csv"
    files_present = glob.glob(file_name)
    if not files_present:
        df.to_csv(file_name, index=False)
    return df


directory = "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/"
if not os.path.exists(directory):
    os.makedirs(directory, exist_ok=True)
directory_cell8 = "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/"
if not os.path.exists(directory_cell8):
    os.makedirs(directory_cell8, exist_ok=True)
directory_cell8 = "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/charts/"
if not os.path.exists(directory_cell8):
    os.makedirs(directory_cell8, exist_ok=True)
base_dir_2017 = "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2017/"
file_name_2017 = "multipleChoiceResponses.csv"
responses_df_2017 = load_survey_data(base_dir_2017, file_name_2017, rows_to_skip=0)
base_dir_2018 = "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2018/"
file_name_2018 = "multipleChoiceResponses.csv"
responses_df_2018 = load_survey_data(base_dir_2018, file_name_2018)
base_dir_2019 = "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2019/"
file_name_2019 = "multiple_choice_responses.csv"
responses_df_2019 = load_survey_data(base_dir_2019, file_name_2019)
base_dir_2020 = "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2020/"
file_name_2020 = "kaggle_survey_2020_responses.csv"
responses_df_2020 = load_survey_data(base_dir_2020, file_name_2020)
base_dir_2021 = "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2021/"
file_name_2021 = "kaggle_survey_2021_responses.csv"
responses_df_2021 = load_survey_data(base_dir_2021, file_name_2021)
base_dir_2022 = "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/input/kaggle-survey-2022/"
file_name_2022 = "kaggle_survey_2022_responses.csv"
responses_df_2022 = load_survey_data(base_dir_2022, file_name_2022)

CPU times: user 3.76 s, sys: 841 ms, total: 4.6 s
Wall time: 4.6 s


# *Step 4: Clean the data and apply filters*
Filter according to job title, country of residence, etc

In [6]:
%%time
### cell 2 ###

responses_df_2018_cell10 = responses_df_2018[
    responses_df_2018.columns.drop(list(responses_df_2018.filter(regex="- Text")))
]
responses_df_2019_cell10 = responses_df_2019[
    responses_df_2019.columns.drop(list(responses_df_2019.filter(regex="- Text")))
]


def replace_hyphen_with_en_dash(df):
    df.columns = df.columns.str.replace("Scikit-learn", "Scikit—learn")
    df.columns = df.columns.str.replace("peer-reviewed", "peer—reviewed")
    df.columns = df.columns.str.replace("Cloud-certification", "Cloud—certification")
    df.columns = df.columns.str.replace("U-Net, Mask R-CNN", "U—Net, Mask R—CNN")
    df.columns = df.columns.str.replace("Encoder-decoder", "Encoder—decoder")
    df.columns = df.columns.str.replace("GPT-3", "GPT—3")
    df.columns = df.columns.str.replace("gpt-3", "gpt—3")
    df.columns = df.columns.str.replace("pre-trained", "pre—trained")
    df.columns = df.columns.str.replace("What-if", "What—if")
    df.columns = df.columns.str.replace("Audit-AI", "Audit—AI")
    return df


responses_df_2022_cell10 = replace_hyphen_with_en_dash(responses_df_2022)

CPU times: user 831 ms, sys: 69.1 ms, total: 901 ms
Wall time: 864 ms


# *Step 5: Preview the data*

`kaggle_survey_2022_responses.csv`: **43 questions and 23,997 responses**

- Responses to multiple choice questions (only a single choice can be selected)
were recorded in individual columns. Responses to multiple selection questions
(multiple choices can be selected) were split into multiple columns (with one
column per answer choice).

`kaggle_survey_2022_answer_choices.pdf`: **list of answer choices for every question**

- With footnotes describing which questions were asked to which respondents.

`kaggle_survey_2022_methodology.pdf`: **a description of how the survey was conducted**

- You can ask additional questions by posting in the pinned Q&A thread.


In [7]:
%%time
### cell 3 ###

responses_df_2022_cell10.head(5)

CPU times: user 29.6 ms, sys: 4.26 ms, total: 33.8 ms
Wall time: 30.6 ms


,Duration (in seconds),What is your age (# years)?,What is your gender? - Selected Choice,In which country do you currently reside?,"Are you currently a student? (high school, university, or graduate)",On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Coursera,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - edX,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Kaggle Learn Courses,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - DataCamp,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Fast.ai,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Udacity,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Udemy,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - LinkedIn Learning,"On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Cloud—certification programs (direct from AWS, Azure, GCP, or similar)",On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - University Courses (resulting in a university degree),On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - None,On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Other,What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - University courses,"What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - Online courses (Coursera, EdX, etc)","What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - Social media platforms (Reddit, Twitter, etc)","What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - Video platforms (YouTube, Twitch, etc)","What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - Kaggle (notebooks, competitions, etc)",What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - None / I do not study data science,What products or platforms did you find to be most helpful when you first started studying data science? (Select all that apply) - Selected Choice - Other,What is the highest level of formal education that you have attained or plan to attain within the next 2 years?,"Have you ever published any academic research (papers, preprints, conference proceedings, etc)?","Did your research make use of machine learning? - Yes, the research made advances related to some novel machine learning method (theoretical research)","Did your research make use of machine learning? - Yes, the research made use of machine learning as a tool (applied research)",Did your research make use of machine learning? - No,For how many years have you been writing code and/or programming?,What programming languages do you use on a regular basis? (Select all that apply) - Selected Choice - Python,What programming languages do you use on a regular basis? (Select all that apply) - Selected Choice - R,What programming languages do you use on a regular basis? (Select all that apply) - Selected Choice - SQL,What programming langu

# *Step 6: Create data visualizations*

In [8]:
%%time
### cell 4 ###


def create_dataframe_of_counts_16(
    dataframe, column, rename_index, rename_column, return_percentages=False
):
    df = dataframe[column].value_counts().reset_index()
    if return_percentages == True:
        df[column] = df[column] * 100 / df[column].sum()
    df = pd.DataFrame(df)
    df = df.rename({"index": rename_index, column: rename_column}, axis="columns")
    return df


percentages_per_country_df = create_dataframe_of_counts_16(
    responses_df_2022_cell10[1:],
    "In which country do you currently reside?",
    "country",
    "% of respondents",
    return_percentages=False,
)
percentages_per_country_df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   % of respondents  58 non-null     object
 1   count             58 non-null     int64
dtypes: int64(1), object(1)
memory usage: 1.2+ KB
CPU times: user 113 ms, sys: 4.78 ms, total: 118 ms
Wall time: 113 ms


In [9]:
%%time
### cell 5 ###


def count_then_return_percent_17(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest = "In which country do you currently reside?"
responses_df_2022_cell10.rename(
    columns={
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom (UK)"
    },
    inplace=True,
)
responses_df_2022_cell10.replace(
    ["United Kingdom of Great Britain and Northern Ireland"],
    "United Kingdom (UK)",
    inplace=True,
)
responses_in_order = [
    "Other",
    "India",
    "United States of America",
    "Brazil",
    "Nigeria",
    "Pakistan",
    "Japan",
    "China",
    "Egypt",
    "Indonesia",
    "Mexico",
    "Turkey",
    "Russia",
]
responses_df_2022_cell10[question_of_interest][
    ~responses_df_2022_cell10[question_of_interest].isin(responses_in_order)
] = "Other"
percentages = count_then_return_percent_17(
    responses_df_2022_cell10, question_of_interest
).sort_index()[responses_in_order]
percentages_cell17 = percentages.sort_values(ascending=True)
orientation_for_chart = "h"
percentages_cell17.info()

<class 'pandas.core.series.Series'>
Index: 13 entries, Turkey to India
Series name: count
Non-Null Count  Dtype  
--------------  -----  
13 non-null     float64
dtypes: float64(1)
memory usage: 208.0+ bytes
CPU times: user 580 ms, sys: 37.5 ms, total: 618 ms
Wall time: 602 ms


In [10]:
%%time
### cell 6 ###


def count_then_return_percent_18(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_18(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_18(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_18(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_18(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_18(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_name_alternate = "Country"
responses_df_2022_cell10.rename(
    columns={
        "United Kingdom (UK)": "United Kingdom of Great Britain and Northern Ireland"
    },
    inplace=True,
)
responses_df_2022_cell10.replace(
    ["United Kingdom (UK)"],
    "United Kingdom of Great Britain and Northern Ireland",
    inplace=True,
)
responses_df_2017[question_name_alternate].replace(
    ["United States"], "United States of America", inplace=True
)
responses_df_2017[question_name_alternate].replace(
    ["People 's Republic of China"], "China", inplace=True
)
responses_df_2017[question_name_alternate].replace(
    ["United Kingdom"],
    "United Kingdom of Great Britain and Northern Ireland",
    inplace=True,
)
responses_df_2017.rename(
    columns={question_name_alternate: "In which country do you currently reside?"},
    inplace=True,
)
question_name_alternate_cell18 = "CurrentJobTitleSelect"
responses_df_2017.rename(
    columns={
        question_name_alternate_cell18: "Select the title most similar to your current role (or most recent title if retired): - Selected Choice"
    },
    inplace=True,
)
subset_of_countries = [
    "Other",
    "India",
    "United States of America",
    "Brazil",
    "Nigeria",
    "Pakistan",
    "Japan",
    "China",
    "Egypt",
    "Indonesia",
    "Mexico",
    "Turkey",
    "Russia",
]
question_name = "In which country do you currently reside?"
responses_df_2017[question_name][
    ~responses_df_2017[question_name].isin(subset_of_countries)
] = "Other"
responses_df_2018_cell10[question_name][
    ~responses_df_2018_cell10[question_name].isin(subset_of_countries)
] = "Other"
responses_df_2019_cell10[question_name][
    ~responses_df_2019_cell10[question_name].isin(subset_of_countries)
] = "Other"
responses_df_2020[question_name][
    ~responses_df_2020[question_name].isin(subset_of_countries)
] = "Other"
responses_df_2021[question_name][
    ~responses_df_2021[question_name].isin(subset_of_countries)
] = "Other"
responses_df_2022_cell10[question_name][
    ~responses_df_2022_cell10[question_name].isin(subset_of_countries)
] = "Other"
question_of_interest_cell18 = "In which country do you currently reside?"
title_for_x_axis = ""
country_df_combined = combine_subset_of_data_from_multiple_years_18(
    question_of_interest_cell18, title_for_x_axis, include_2017="yes"
)
country_df_combined_cell18 = country_df_combined.sort_values(
    by=["year", "percentage"], ascending=True
)
country_df_combined_cell18.rename(
    columns={
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom (UK)"
    },
    inplace=True,
)
country_df_combined_cell18.replace(
    ["United Kingdom of Great Britain and Northern Ireland"],
    "United Kingdom",
    inplace=True,
)
country_df_combined_cell18.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 309 entries, Egypt to India
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   percentage  309 non-null    float64
 1   year        309 non-null    object
 2               309 non-null    object
dtypes: float64(1), object(2)
memory usage: 12.7+ KB
CPU times: user 1.19 s, sys: 94.1 ms, total: 1.29 s
Wall time: 1.23 s


In [11]:
%%time
### cell 7 ###


def count_then_return_percent_19(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell19 = "What is your age (# years)?"
percentages_cell19 = count_then_return_percent_19(
    responses_df_2022_cell10, question_of_interest_cell19
).sort_index()
percentages_cell19.info()

<class 'pandas.core.series.Series'>
Index: 11 entries, 18-21 to 70+
Series name: count
Non-Null Count  Dtype  
--------------  -----  
11 non-null     float64
dtypes: float64(1)
memory usage: 176.0+ bytes
CPU times: user 89.1 ms, sys: 4.72 ms, total: 93.8 ms
Wall time: 87.7 ms


In [12]:
%%time
### cell 8 ###


def count_then_return_percent_20(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_20(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_20(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_20(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_20(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_20(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell20 = "What is your age (# years)?"
title_for_x_axis_cell20 = ""
responses_df_2018_cell10[question_of_interest_cell20].replace(
    ["70-79", "80+"], "70+", inplace=True
)
age_df_combined = combine_subset_of_data_from_multiple_years_20(
    question_of_interest_cell20, title_for_x_axis_cell20
)
age_df_combined.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 56 entries, 18-21 to 70+
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   percentage  56 non-null     float64
 1   year        56 non-null     object
 2               56 non-null     object
dtypes: float64(1), object(2)
memory usage: 1.9+ KB
CPU times: user 424 ms, sys: 49.8 ms, total: 474 ms
Wall time: 448 ms


In [13]:
%%time
### cell 9 ###


def count_then_return_percent_21(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell21 = "What is your gender? - Selected Choice"
responses_in_order_cell21 = [
    "Man",
    "Woman",
    "Nonbinary",
    "Prefer to self-describe",
    "Prefer not to say",
]
percentages_cell21 = (
    count_then_return_percent_21(responses_df_2022_cell10, question_of_interest_cell21)
    .sort_index()[responses_in_order_cell21]
    .iloc[::-1]
)
percentages_cell21.info()

<class 'pandas.core.series.Series'>
Index: 5 entries, Prefer not to say to Man
Series name: count
Non-Null Count  Dtype  
--------------  -----  
5 non-null      float64
dtypes: float64(1)
memory usage: 80.0+ bytes
CPU times: user 98.8 ms, sys: 13 ms, total: 112 ms
Wall time: 102 ms


In [14]:
%%time
### cell 10 ###


def count_then_return_percent_22(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_22(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_22(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_22(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_22(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_22(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


responses_df_2017["GenderSelect"].replace(["Male"], "Man", inplace=True)
responses_df_2017["GenderSelect"].replace(["Female"], "Woman", inplace=True)
responses_df_2017["GenderSelect"].replace(
    ["A different identity", "Non-binary, genderqueer, or gender non-conforming"],
    "Prefer to self-describe",
    inplace=True,
)
responses_df_2017["GenderSelect"].replace(
    ["Non-binary, genderqueer, or gender non-conforming"], "Nonbinary", inplace=True
)
responses_df_2018_cell10["What is your gender? - Selected Choice"].replace(
    ["Male"], "Man", inplace=True
)
responses_df_2018_cell10["What is your gender? - Selected Choice"].replace(
    ["Female"], "Woman", inplace=True
)
responses_df_2019_cell10["What is your gender? - Selected Choice"].replace(
    ["Male"], "Man", inplace=True
)
responses_df_2019_cell10["What is your gender? - Selected Choice"].replace(
    ["Female"], "Woman", inplace=True
)
responses_df_2017["GenderSelect"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
responses_df_2017.columns = responses_df_2017.columns.str.replace(
    "GenderSelect", "What is your gender? - Selected Choice", regex=False
)
responses_df_2018_cell10["What is your gender? - Selected Choice"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
responses_df_2019_cell10["What is your gender? - Selected Choice"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
responses_df_2020["What is your gender? - Selected Choice"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
responses_df_2021["What is your gender? - Selected Choice"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
responses_df_2022_cell10["What is your gender? - Selected Choice"].replace(
    ["Nonbinary", "Prefer not to say"], "Prefer to self-describe", inplace=True
)
question_of_interest_cell22 = "What is your gender? - Selected Choice"
title_for_x_axis_cell22 = ""
age_df_combined_cell22 = combine_subset_of_data_from_multiple_years_22(
    question_of_interest_cell22, title_for_x_axis_cell22, include_2017="yes"
)
age_df_combined_cell22 = age_df_combined.sort_values(
    by=["year", "percentage"], ascending=True
)
age_df_combined_cell22.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 56 entries, 70-79 to 18-21
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   percentage  56 non-null     float64
 1   year        56 non-null     object
 2               56 non-null     object
dtypes: float64(1), object(2)
memory usage: 1.9+ KB
CPU times: user 743 ms, sys: 58.4 ms, total: 801 ms
Wall time: 754 ms


In [15]:
%%time
### cell 11 ###


def count_then_return_percent_23(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell23 = (
    "Are you currently a student? (high school, university, or graduate)"
)
percentages_cell23 = count_then_return_percent_23(
    responses_df_2022_cell10, question_of_interest_cell23
).sort_index()
percentages_cell23

CPU times: user 74 ms, sys: 15.2 ms, total: 89.2 ms
Wall time: 84 ms


Are you currently a student? (high school, university, or graduate)
No     50.2
Yes    49.8
Name: count, dtype: float64

In [16]:
%%time
### cell 12 ###


def bar_chart_multiple_choice_24(
    response_counts, title, y_axis_title, orientation="h", num_choices=15
):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ""
    pd.DataFrame(response_counts_series).to_csv(
        "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/"
        + title
        + ".csv",
        index=True,
    )
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)


def count_then_return_percent_24(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


USA_responses_df_2022 = responses_df_2022_cell10[
    responses_df_2022_cell10["In which country do you currently reside?"]
    == "United States of America"
]
question_of_interest_cell24 = (
    "Are you currently a student? (high school, university, or graduate)"
)
percentages_cell24 = count_then_return_percent_24(
    USA_responses_df_2022, question_of_interest_cell24
).sort_index()
title_for_chart_cell24 = "Students status for Kaggle Survey participants from the USA"
title_for_y_axis_cell24 = "% of respondents"
bar_chart_multiple_choice_24(
    response_counts=percentages_cell24,
    title=title_for_chart_cell24,
    y_axis_title=title_for_y_axis_cell24,
    orientation=orientation_for_chart,
)
India_responses_df_2022 = responses_df_2022_cell10[
    responses_df_2022_cell10["In which country do you currently reside?"] == "India"
]
question_of_interest_cell24 = (
    "Are you currently a student? (high school, university, or graduate)"
)
percentages_cell24 = count_then_return_percent_24(
    India_responses_df_2022, question_of_interest_cell24
).sort_index()
title_for_chart_cell24 = "Students status for Kaggle Survey participants from India"
title_for_y_axis_cell24 = "% of respondents"
percentages_cell24.info()

<class 'pandas.core.series.Series'>
Index: 2 entries, No to Yes
Series name: count
Non-Null Count  Dtype  
--------------  -----  
2 non-null      float64
dtypes: float64(1)
memory usage: 32.0+ bytes
CPU times: user 242 ms, sys: 48.9 ms, total: 290 ms
Wall time: 275 ms


In [17]:
%%time
### cell 13 ###


def grab_subset_of_data_25(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell25 = (
    "On which platforms have you begun or completed data science courses?"
)
online_learning_platforms_df_2022 = grab_subset_of_data_25(
    responses_df_2022_cell10, question_of_interest_cell25
)
online_learning_platforms_df_2022.columns = (
    online_learning_platforms_df_2022.columns.str.replace(
        "(direct from AWS, Azure, GCP, or similar)", "", regex=False
    )
)
online_learning_platforms_df_2022.columns = (
    online_learning_platforms_df_2022.columns.str.replace(
        "(resulting in a university degree)", "", regex=False
    )
)
online_learning_platforms_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21354 entries, 0 to 23996
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Coursera                       9699 non-null   object 
 1   edX                            2474 non-null   object 
 2   Kaggle Learn Courses           6628 non-null   object 
 3   DataCamp                       3718 non-null   object 
 4   Fast.ai                        944 non-null    object 
 5   Udacity                        2199 non-null   object 
 6   Udemy                          6116 non-null   object 
 7   LinkedIn Learning              2766 non-null   object 
 8   Cloud—certification programs   1821 non-null   object 
 9   University Courses             6780 non-null   object 
 10  None                           0 non-null      float64
 11  Other                          5669 non-null   object 
dtypes: float64(1), object(11)
memory usage: 2.1+ MB
CPU

In [18]:
%%time
### cell 14 ###


def grab_subset_of_data_26(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_26(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_26(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_26(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_26(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_26(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_26(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_26(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_26(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_26(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_26(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_26(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_26(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_26(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_26(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_26(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_cell26 = (
    "On which platforms have you begun or completed data science courses"
)
question_of_interest_alternate = (
    "On which online platforms have you begun or completed data science courses"
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    question_of_interest_alternate, question_of_interest_cell26
)
responses_df_2018_cell10.replace(["Kaggle Learn"], "Kaggle Learn Courses", inplace=True)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Kaggle Learn", "Kaggle Learn Courses", regex=False
)
responses_df_2018_cell10.replace(["Fast.AI"], "Fast.ai", inplace=True)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Fast.AI", "Fast.ai"
)
responses_df_2018_cell10.replace(
    ["Online University Courses"],
    "University Courses (resulting in a university degree)",
    inplace=True,
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Online University Courses",
    "University Courses (resulting in a university degree)",
    regex=False,
)
responses_df_2019_cell10.replace(
    ["Kaggle Courses (i.e. Kaggle Learn)"], "Kaggle Learn Courses", inplace=True
)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    "Kaggle Courses (i.e. Kaggle Learn)", "Kaggle Learn Courses", regex=False
)
question_of_interest_cell26 = (
    "On which platforms have you begun or completed data science courses?"
)
(learning_platform_df_combined, learning_platform_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(
        question_of_interest_cell26
    )
)
learning_platform_df_combined_percentages = convert_df_of_counts_to_percentages_26(
    learning_platform_df_combined, learning_platform_df_combined_counts
)
learning_platform_df_combined_percentages.columns = (
    learning_platform_df_combined_percentages.columns.str.replace(
        "(resulting in a university degree)", "", regex=False
    )
)
learning_platform_df_combined_percentages_cell26 = (
    learning_platform_df_combined_percentages.loc[
        :,
        [
            "year",
            "Coursera",
            "University Courses ",
            "Kaggle Learn Courses",
            "Udemy",
            "Udacity",
            "DataCamp",
            "edX",
            "Fast.ai",
            "None",
            "Other",
        ],
    ]
)
df = learning_platform_df_combined_percentages_cell26.melt(
    id_vars=["year"],
    value_vars=[
        "Coursera",
        "University Courses ",
        "Kaggle Learn Courses",
        "Udemy",
        "Udacity",
        "DataCamp",
        "edX",
        "Fast.ai",
    ],
)
df_cell26 = df.sort_values(by=["year", "value"], ascending=True)
df_cell26.columns = df_cell26.columns.str.replace("variable", "")
df_cell26.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 40 entries, 35 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    40 non-null     object
 1           40 non-null     object
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.5+ KB
CPU times: user 2.7 s, sys: 288 ms, total: 2.98 s
Wall time: 2.93 s


In [19]:
%%time
### cell 15 ###


def grab_subset_of_data_27(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell27 = "What products or platforms did you find to be most helpful when you first started studying data science?"
online_learning_platforms_df_2022_cell27 = grab_subset_of_data_27(
    responses_df_2022_cell10, question_of_interest_cell27
)
online_learning_platforms_df_2022_cell27.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23696 entries, 1 to 23996
Data columns (total 7 columns):
 #   Column                                         Non-Null Count  Dtype 
---  ------                                         --------------  ----- 
 0   University courses                             6851 non-null   object
 1   Online courses (Coursera, EdX, etc)            13714 non-null  object
 2   Social media platforms (Reddit, Twitter, etc)  3310 non-null   object
 3   Video platforms (YouTube, Twitch, etc)         12871 non-null  object
 4   Kaggle (notebooks, competitions, etc)          12700 non-null  object
 5   None / I do not study data science             1022 non-null   object
 6   Other                                          1944 non-null   object
dtypes: object(7)
memory usage: 1.4+ MB
CPU times: user 147 ms, sys: 24 ms, total: 171 ms
Wall time: 165 ms


In [20]:
%%time
### cell 16 ###


def count_then_return_percent_28(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell28 = "What is the highest level of formal education that you have attained or plan to attain within the next 2 years?"
responses_df_2022_cell10[question_of_interest_cell28].replace(
    ["Bachelorâ\x80\x99s degree"], "Bachelor's degree", inplace=True
)
responses_df_2022_cell10[question_of_interest_cell28].replace(
    ["Masterâ\x80\x99s degree"], "Master's degree", inplace=True
)
responses_df_2022_cell10[question_of_interest_cell28].replace(
    ["Some college/university study without earning a bachelorâ\x80\x99s degree"],
    "Some university study but without earning a degree",
    inplace=True,
)
responses_in_order_cell28 = [
    "No formal education past high school",
    "Some university study but without earning a degree",
    "Bachelor's degree",
    "Master's degree",
    "Doctoral degree",
    "Professional doctorate",
]
percentages_cell28 = count_then_return_percent_28(
    responses_df_2022_cell10, question_of_interest_cell28
).sort_index()[responses_in_order_cell28]
percentages_cell28.info()

<class 'pandas.core.series.Series'>
Index: 6 entries, No formal education past high school to Professional doctorate
Series name: count
Non-Null Count  Dtype  
--------------  -----  
6 non-null      float64
dtypes: float64(1)
memory usage: 96.0+ bytes
CPU times: user 171 ms, sys: 4.29 ms, total: 176 ms
Wall time: 167 ms


In [21]:
%%time
### cell 17 ###


def count_then_return_percent_29(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_29(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_29(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_29(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_29(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_29(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell29 = "What is the highest level of formal education that you have attained or plan to attain within the next 2 years?"
responses_df_2017.rename(
    columns={"FormalEducation": question_of_interest_cell29}, inplace=True
)
responses_df_2017[question_of_interest_cell29].replace(
    ["Master's degree"], "Master's degree", inplace=True
)
responses_df_2017[question_of_interest_cell29].replace(
    ["Bachelor's degree"], "Bachelor's degree", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell29].replace(
    ["Masterâ\x80\x99s degree"], "Master's degree", inplace=True
)
responses_df_2019_cell10[question_of_interest_cell29].replace(
    ["Masterâ\x80\x99s degree"], "Master's degree", inplace=True
)
responses_df_2020[question_of_interest_cell29].replace(
    ["Masterâ\x80\x99s degree"], "Master's degree", inplace=True
)
responses_df_2021[question_of_interest_cell29].replace(
    ["Masterâ\x80\x99s degree"], "Master's degree", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell29].replace(
    ["Bachelorâ\x80\x99s degree"], "Bachelor's degree", inplace=True
)
responses_df_2019_cell10[question_of_interest_cell29].replace(
    ["Bachelorâ\x80\x99s degree"], "Bachelor's degree", inplace=True
)
responses_df_2020[question_of_interest_cell29].replace(
    ["Bachelorâ\x80\x99s degree"], "Bachelor's degree", inplace=True
)
responses_df_2021[question_of_interest_cell29].replace(
    ["Bachelorâ\x80\x99s degree"], "Bachelor's degree", inplace=True
)
question_of_interest_cell29 = "What is the highest level of formal education that you have attained or plan to attain within the next 2 years?"
title_for_x_axis_cell29 = ""
degree_df_combined = combine_subset_of_data_from_multiple_years_29(
    question_of_interest_cell29, title_for_x_axis_cell29, include_2017="yes"
)
degree_df_combined_cell29 = degree_df_combined.sort_values(
    by=["year", "percentage"], ascending=True
)
subset_of_degrees = ["Bachelor's degree", "Master's degree", "Doctoral degree"]
degree_df_combined_cell29[""][
    ~degree_df_combined_cell29[""].isin(subset_of_degrees)
] = "Other"
degree_df_combined_cell29.info()

<class 'pandas.core.frame.DataFrame'>
Index: 47 entries, I prefer not to answer to Master's degree
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  47 non-null     float64
 1   year        47 non-null     object 
 2               47 non-null     object 
dtypes: float64(1), object(2)
memory usage: 2.5+ KB
CPU times: user 633 ms, sys: 27 ms, total: 660 ms
Wall time: 626 ms


In [22]:
%%time
### cell 18 ###


def count_then_return_percent_30(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell30 = "Have you ever published any academic research (papers, preprints, conference proceedings, etc)?"
percentages_cell30 = count_then_return_percent_30(
    responses_df_2022_cell10, question_of_interest_cell30
).sort_index()
percentages_cell30.info()

<class 'pandas.core.series.Series'>
Index: 3 entries, No to nan
Series name: count
Non-Null Count  Dtype  
--------------  -----  
3 non-null      float64
dtypes: float64(1)
memory usage: 156.0+ bytes
CPU times: user 71.8 ms, sys: 0 ns, total: 71.8 ms
Wall time: 67.4 ms


In [23]:
%%time
### cell 19 ###


def grab_subset_of_data_31(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell31 = "Did your research make use of machine learning?"
online_learning_platforms_df_2022_cell31 = grab_subset_of_data_31(
    responses_df_2022_cell10, question_of_interest_cell31
)
online_learning_platforms_df_2022_cell31.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5223 entries, 1 to 23995
Data columns (total 3 columns):
 #   Column                                                                                                Non-Null Count  Dtype 
---  ------                                                                                                --------------  ----- 
 0   Yes, the research made advances related to some novel machine learning method (theoretical research)  1549 non-null   object
 1   Yes, the research made use of machine learning as a tool (applied research)                           2463 non-null   object
 2   No                                                                                                    1913 non-null   object
dtypes: object(3)
memory usage: 163.2+ KB
CPU times: user 143 ms, sys: 27.5 ms, total: 171 ms
Wall time: 165 ms


In [24]:
%%time
### cell 20 ###


def count_then_return_percent_32(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell32 = (
    "For how many years have you been writing code and/or programming?"
)
responses_in_order_cell32 = [
    "I have never written code",
    "< 1 years",
    "1-3 years",
    "3-5 years",
    "5-10 years",
    "10-20 years",
    "20+ years",
]
percentages_cell32 = count_then_return_percent_32(
    responses_df_2022_cell10, question_of_interest_cell32
).sort_index()[responses_in_order_cell32]
percentages_cell32.info()

<class 'pandas.core.series.Series'>
Index: 7 entries, I have never written code to 20+ years
Series name: count
Non-Null Count  Dtype  
--------------  -----  
7 non-null      float64
dtypes: float64(1)
memory usage: 112.0+ bytes
CPU times: user 72.6 ms, sys: 515 μs, total: 73.1 ms
Wall time: 68.9 ms


In [25]:
%%time
### cell 21 ###


def count_then_return_percent_33(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_33(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_33(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_33(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_33(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_33(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell33 = (
    "For how many years have you been writing code and/or programming?"
)
responses_df_2018_cell10.rename(
    columns={
        "How long have you been writing code to analyze data?": question_of_interest_cell33
    },
    inplace=True,
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    ["< 1 year"], "< 1 years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    [
        "I have never written code but I want to learn",
        "I have never written code and I do not want to learn",
    ],
    "I have never written code",
    inplace=True,
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    ["1-2 years"], "1-3 years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    ["20-30 years"], "20+ years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    ["30-40 years"], "20+ years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell33].replace(
    ["40+ years"], "20+ years", inplace=True
)
responses_df_2019_cell10.rename(
    columns={
        "How long have you been writing code to analyze data (at work or at school)?": question_of_interest_cell33
    },
    inplace=True,
)
responses_df_2019_cell10[question_of_interest_cell33].replace(
    ["1-2 years"], "1-3 years", inplace=True
)
responses_df_2020[question_of_interest_cell33].replace(
    ["1-2 years"], "1-3 years", inplace=True
)
title_for_x_axis_cell33 = ""
programming_df_combined = combine_subset_of_data_from_multiple_years_33(
    question_of_interest_cell33, title_for_x_axis_cell33
).sort_values(by=["year", "percentage"], ascending=True)
programming_df_combined.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39 entries, 20+ years to 1-3 years
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  39 non-null     float64
 1   year        39 non-null     object 
 2               35 non-null     object 
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 598 ms, sys: 17.9 ms, total: 616 ms
Wall time: 589 ms


In [26]:
%%time
### cell 22 ###


def grab_subset_of_data_34(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell34 = (
    "What programming languages do you use on a regular basis?"
)
online_learning_platforms_df_2022_cell34 = grab_subset_of_data_34(
    responses_df_2022_cell10, question_of_interest_cell34
)
online_learning_platforms_df_2022_cell34.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20879 entries, 1 to 23996
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Python      18653 non-null  object 
 1   R           4571 non-null   object 
 2   SQL         9620 non-null   object 
 3   C           3801 non-null   object 
 4   C#          1473 non-null   object 
 5   C++         4549 non-null   object 
 6   Java        3862 non-null   object 
 7   Javascript  3489 non-null   object 
 8   Bash        1674 non-null   object 
 9   PHP         1443 non-null   object 
 10  MATLAB      2441 non-null   object 
 11  Julia       296 non-null    object 
 12  Go          322 non-null    object 
 13  None        0 non-null      float64
 14  Other       1342 non-null   object 
dtypes: float64(1), object(14)
memory usage: 2.5+ MB
CPU times: user 166 ms, sys: 16.3 ms, total: 183 ms
Wall time: 176 ms


In [27]:
%%time
### cell 23 ###


def grab_subset_of_data_35(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_35(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_35(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_35(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_35(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_35(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_35(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_35(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_35(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_35(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_35(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_35(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_35(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_35(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_35(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_35(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_cell35 = (
    "What programming languages do you use on a regular basis?"
)
(language_df_combined, language_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest_cell35
    )
)
language_df_combined_percentages = convert_df_of_counts_to_percentages_35(
    language_df_combined, language_df_combined_counts
)
language_df_combined_percentages_cell35 = language_df_combined_percentages.loc[
    :, ["year", "Python", "SQL", "C++", "C", "R", "Java", "Javascript", "Other"]
]
df_cell35 = language_df_combined_percentages_cell35.melt(
    id_vars=["year"],
    value_vars=["Python", "SQL", "C++", "C", "R", "Java", "Javascript", "Other"],
)
df_cell35 = df.sort_values(by=["year", "value"], ascending=True)
df_cell35 = df.rename(columns={"variable": ""})
df_cell35.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    40 non-null     object
 1           40 non-null     object
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 1.18 s, sys: 101 ms, total: 1.28 s
Wall time: 1.25 s


In [28]:
%%time
### cell 24 ###


def grab_subset_of_data_36(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell36 = "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
ide_df_2022 = grab_subset_of_data_36(
    responses_df_2022_cell10, question_of_interest_cell36
)
ide_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20183 entries, 1 to 23996
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   JupyterLab                    4887 non-null   object 
 1   RStudio                       3824 non-null   object 
 2   Visual Studio                 4416 non-null   object 
 3   Visual Studio Code (VSCode)   9976 non-null   object 
 4   PyCharm                       6099 non-null   object 
 5   Spyder                        2880 non-null   object 
 6   Notepad++                     3891 non-null   object 
 7   Sublime Text                  2218 non-null   object 
 8   Vim / Emacs                   1448 non-null   object 
 9   MATLAB                        2302 non-null   object 
 10  Jupyter Notebook              13684 non-null  object 
 11  IntelliJ                      1612 non-null   object 
 12  None                          0 non-null      float64
 13  Other 

In [29]:
%%time
### cell 25 ###


def grab_subset_of_data_37(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell37 = "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
ide_df_2022_cell37 = grab_subset_of_data_37(
    responses_df_2022_cell10, question_of_interest_cell37
)
columns_to_combine = ["Jupyter Notebook", "JupyterLab "]
ide_df_2022_cell37["Jupyter/JupyterLab"] = (
    ide_df_2022_cell37[columns_to_combine].notna().any(axis="columns")
)
ide_df_2022_cell37 = ide_df_2022_cell37.drop(columns=columns_to_combine)
ide_df_2022_cell37["Jupyter/JupyterLab"].replace(
    [True], "Jupyter/JupyterLab", inplace=True
)
ide_df_2022_cell37["Jupyter/JupyterLab"].replace([False], np.nan, inplace=True)
columns_to_combine_cell37 = ["Visual Studio Code (VSCode) ", "Visual Studio "]
ide_df_2022_cell37["Visual Studio / Visual Studio Code (VSCode)"] = (
    ide_df_2022_cell37[columns_to_combine_cell37].notna().any(axis="columns")
)
ide_df_2022_cell37 = ide_df_2022_cell37.drop(columns=columns_to_combine_cell37)
ide_df_2022_cell37["Visual Studio / Visual Studio Code (VSCode)"].replace(
    [True], "Visual Studio / Visual Studio Code (VSCode)", inplace=True
)
ide_df_2022_cell37["Visual Studio / Visual Studio Code (VSCode)"].replace(
    [False], np.nan, inplace=True
)
ide_df_2022_cell37.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20183 entries, 1 to 23996
Data columns (total 12 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   RStudio                                      3824 non-null   object 
 1   PyCharm                                      6099 non-null   object 
 2   Spyder                                       2880 non-null   object 
 3   Notepad++                                    3891 non-null   object 
 4   Sublime Text                                 2218 non-null   object 
 5   Vim / Emacs                                  1448 non-null   object 
 6   MATLAB                                       2302 non-null   object 
 7   IntelliJ                                     1612 non-null   object 
 8   None                                         0 non-null      float64
 9   Other                                        1474 non-null   object 
 10  Jup

In [30]:
print(responses_df_2022_cell10.columns)
print(responses_df_2021.columns)
print(responses_df_2020.columns)
print(responses_df_2019_cell10.columns)
print(responses_df_2018_cell10.columns)

Index(['Duration (in seconds)', 'What is your age (# years)?',
       'What is your gender? - Selected Choice',
       'In which country do you currently reside?',
       'Are you currently a student? (high school, university, or graduate)',
       'On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Coursera',
       'On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - edX',
       'On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Kaggle Learn Courses',
       'On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - DataCamp',
       'On which platforms have you begun or completed data science courses? (Select all that apply) - Selected Choice - Fast.ai',
       ...
       'Who/what are your favorite media sources that report on data science topi

In [31]:
%%time
### cell 26 ###


def grab_subset_of_data_38(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_38(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_38(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_38(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_38(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_38(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_38(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_38(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_38(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_38(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_38(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_38(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_38(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_38(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_38(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_38(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


correct_phrasing = "Jupyter Notebook / JupyterLab"
incorrect_phrasing = "Jupyter/IPython"
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    incorrect_phrasing, correct_phrasing, regex=False
)
question_of_interest_cell38 = "Which of the following integrated development environments (IDE's) do you use on a regular basis?"
alternate_phrasing = "Which of the following integrated development environments (IDE's) have you used at work or school in the last 5 years?"
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    alternate_phrasing, question_of_interest_cell38, regex=False
)
x_axis_title = "Percentage of respondents"
(ide_df_combined, ide_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(
        question_of_interest_cell38
    )
)
ide_df_combined_2 = ide_df_combined.copy()
columns_to_combine_cell38 = [
    "Jupyter Notebook",
    "JupyterLab ",
    "Jupyter (JupyterLab, Jupyter Notebooks, etc) ",
    "Jupyter Notebook / JupyterLab",
]
ide_df_combined_2["Jupyter Notebook / JupyterLab_"] = (
    ide_df_combined_2[columns_to_combine_cell38].notna().any(axis="columns")
)
ide_df_combined_2_cell38 = ide_df_combined_2.drop(columns=columns_to_combine_cell38)
ide_df_combined_2_cell38["Jupyter Notebook / JupyterLab_"].replace(
    [True], "Jupyter Notebook / JupyterLab", inplace=True
)
ide_df_combined_2_cell38["Jupyter Notebook / JupyterLab_"].replace(
    [False], np.nan, inplace=True
)
ide_df_combined_2_cell38.columns = ide_df_combined_2_cell38.columns.str.replace(
    "Jupyter Notebook / JupyterLab_", "Jupyter Notebook / JupyterLab", regex=False
)
columns_to_combine_cell38 = ["MATLAB", "MATLAB "]
ide_df_combined_2_cell38["MATLAB_"] = (
    ide_df_combined_2_cell38[columns_to_combine_cell38].notna().any(axis="columns")
)
ide_df_combined_2_cell38 = ide_df_combined_2_cell38.drop(
    columns=columns_to_combine_cell38
)
ide_df_combined_2_cell38["MATLAB_"].replace([True], "MATLAB", inplace=True)
ide_df_combined_2_cell38["MATLAB_"].replace([False], np.nan, inplace=True)
ide_df_combined_2_cell38.columns = ide_df_combined_2_cell38.columns.str.replace(
    "MATLAB_", "MATLAB", regex=False
)
columns_to_combine_cell38 = ["RStudio", "RStudio "]
ide_df_combined_2_cell38["RStudio_"] = (
    ide_df_combined_2_cell38[columns_to_combine_cell38].notna().any(axis="columns")
)
ide_df_combined_2_cell38 = ide_df_combined_2_cell38.drop(
    columns=columns_to_combine_cell38
)
ide_df_combined_2_cell38["RStudio_"].replace([True], "RStudio", inplace=True)
ide_df_combined_2_cell38["RStudio_"].replace([False], np.nan, inplace=True)
ide_df_combined_2_cell38.columns = ide_df_combined_2_cell38.columns.str.replace(
    "RStudio_", "RStudio", regex=False
)
columns_to_combine_cell38 = [
    "Visual Studio Code",
    "Visual Studio",
    "Visual Studio / Visual Studio Code ",
    "Visual Studio ",
    "Visual Studio Code (VSCode) ",
    "Click to write Choice 13",
]
ide_df_combined_2_cell38["Visual Studio / Visual Studio Code (VSCode)"] = (
    ide_df_combined_2_cell38[columns_to_combine_cell38].notna().any(axis="columns")
)
ide_df_combined_2_cell38 = ide_df_combined_2_cell38.drop(
    columns=columns_to_combine_cell38
)
ide_df_combined_2_cell38["Visual Studio / Visual Studio Code (VSCode)"].replace(
    [True], "Visual Studio / Visual Studio Code (VSCode)", inplace=True
)
ide_df_combined_2_cell38["Visual Studio / Visual Studio Code (VSCode)"].replace(
    [False], np.nan, inplace=True
)
columns_to_combine_cell38 = ["PyCharm ", "PyCharm"]
ide_df_combined_2_cell38["PyCharm_"] = (
    ide_df_combined_2_cell38[columns_to_combine_cell38].notna().any(axis="columns")
)
ide_df_combined_2_cell38 = ide_df_combined_2_cell38.drop(
    columns=columns_to_combine_cell38
)
ide_df_combined_2_cell38["PyCharm_"].replace([True], "PyCharm", inplace=True)
ide_df_combined_2_cell38["PyCharm_"].replace([False], np.nan, inplace=True)
ide_df_combined_2_cell38.columns = ide_df_combined_2_cell38.columns.str.replace(
    "PyCharm_", "PyCharm", regex=False
)
ide_df_combined_counts_2 = (
    ide_df_combined_2_cell38.groupby("year").count().reset_index()
)
ide_df_combined_percentages = convert_df_of_counts_to_percentages_38(
    ide_df_combined_2_cell38, ide_df_combined_counts_2
)
ide_df_combined_percentages_cell38 = ide_df_combined_percentages.loc[
    :,
    [
        "year",
        "Jupyter Notebook / JupyterLab",
        "Visual Studio / Visual Studio Code (VSCode)",
        "RStudio",
        "PyCharm",
        "MATLAB",
    ],
]
df_cell38 = ide_df_combined_percentages_cell38.melt(
    id_vars=["year"],
    value_vars=[
        "Jupyter Notebook / JupyterLab",
        "Visual Studio / Visual Studio Code (VSCode)",
        "RStudio",
        "PyCharm",
        "MATLAB",
    ],
)
df_cell38 = df.sort_values(by=["year", "value"], ascending=True)
df_cell38 = df.rename(columns={"variable": ""})
df_cell38.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    40 non-null     object
 1           40 non-null     object
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 1.68 s, sys: 179 ms, total: 1.86 s
Wall time: 1.82 s


In [32]:
%%time
### cell 27 ###


def grab_subset_of_data_39(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell39 = (
    "Do you use any of the following hosted notebook products?"
)
notebooks_df_2022 = grab_subset_of_data_39(
    responses_df_2022_cell10, question_of_interest_cell39
)
notebooks_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13407 entries, 2 to 23996
Data columns (total 16 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Kaggle Notebooks                     7478 non-null   object 
 1   Colab Notebooks                      8929 non-null   object 
 2   Azure Notebooks                      885 non-null    object 
 3   Code Ocean                           158 non-null    object 
 4   IBM Watson Studio                    964 non-null    object 
 5   Amazon Sagemaker Studio              735 non-null    object 
 6   Amazon Sagemaker Studio Lab          545 non-null    object 
 7   Amazon EMR Notebooks                 260 non-null    object 
 8   Google Cloud Vertex AI Workbench     870 non-null    object 
 9   Hex Workspaces                       77 non-null     object 
 10  Noteable Notebooks                   299 non-null    object 
 11  Databricks Collaborative Notebook

In [33]:
%%time
### cell 28 ###


def grab_subset_of_data_40(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_40(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_40(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_40(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_40(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_40(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_40(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_40(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_40(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_40(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_40(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_40(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_40(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_40(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_40(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_40(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_original = (
    "Which of the following hosted notebook products do you use on a regular basis?"
)
question_of_interest_new = "Do you use any of the following hosted notebook products?"
responses_df_2021.columns = responses_df_2021.columns.str.replace(
    question_of_interest_original, question_of_interest_new, regex=False
)
question_of_interest_original_cell40 = (
    "Which of the following hosted notebook products do you use on a regular basis?"
)
question_of_interest_new_cell40 = (
    "Do you use any of the following hosted notebook products?"
)
responses_df_2020.columns = responses_df_2020.columns.str.replace(
    question_of_interest_original_cell40, question_of_interest_new_cell40, regex=False
)
colab_text_to_replace = "Google Colab "
colab_new_text = "Colab Notebooks"
colab_answer = "Which of the following hosted notebook products do you use on a regular basis?  (Select all that apply) - Selected Choice -  Google Colab "
kaggle_text_to_replace = "Kaggle Notebooks (Kernels) "
kaggle_new_text = "Kaggle Notebooks"
kaggle_answer = "Which of the following hosted notebook products do you use on a regular basis?  (Select all that apply) - Selected Choice -  Kaggle Notebooks (Kernels) "
responses_df_2019_cell10[colab_answer] = responses_df_2019_cell10[
    colab_answer
].str.replace(colab_text_to_replace, colab_new_text, regex=False)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    colab_text_to_replace, colab_new_text, regex=False
)
responses_df_2019_cell10[kaggle_answer] = responses_df_2019_cell10[
    kaggle_answer
].str.replace(kaggle_text_to_replace, kaggle_new_text, regex=False)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    kaggle_text_to_replace, kaggle_new_text, regex=False
)
question_of_interest_original_cell40 = (
    "Which of the following hosted notebook products do you use on a regular basis?"
)
question_of_interest_new_cell40 = (
    "Do you use any of the following hosted notebook products?"
)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    question_of_interest_original_cell40, question_of_interest_new_cell40, regex=False
)
colab_text_to_replace_cell40 = "Google Colab"
colab_new_text_cell40 = "Colab Notebooks"
colab_answer_cell40 = "Which of the following hosted notebooks have you used at work or school in the last 5 years? (Select all that apply) - Selected Choice - Google Colab"
kaggle_text_to_replace_cell40 = "Kaggle Kernels"
kaggle_new_text_cell40 = "Kaggle Notebooks"
kaggle_answer_cell40 = "Which of the following hosted notebooks have you used at work or school in the last 5 years? (Select all that apply) - Selected Choice - Kaggle Kernels"
responses_df_2018_cell10[colab_answer_cell40] = responses_df_2018_cell10[
    colab_answer_cell40
].str.replace(colab_text_to_replace_cell40, colab_new_text_cell40, regex=False)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    colab_text_to_replace_cell40, colab_new_text_cell40, regex=False
)
responses_df_2018_cell10[kaggle_answer_cell40] = responses_df_2018_cell10[
    kaggle_answer_cell40
].str.replace(kaggle_text_to_replace_cell40, kaggle_new_text_cell40, regex=False)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    kaggle_text_to_replace_cell40, kaggle_new_text_cell40, regex=False
)
question_of_interest_original_cell40 = "Which of the following hosted notebooks have you used at work or school in the last 5 years?"
question_of_interest_new_cell40 = (
    "Do you use any of the following hosted notebook products?"
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    question_of_interest_original_cell40, question_of_interest_new_cell40, regex=False
)
question_of_interest_cell40 = (
    "Do you use any of the following hosted notebook products?"
)
(notebooks_df_combined, notebooks_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
        question_of_interest_cell40
    )
)
notebooks_df_combined_percentages = convert_df_of_counts_to_percentages_40(
    notebooks_df_combined, notebooks_df_combined_counts
)
notebooks_df_combined_percentages_cell40 = notebooks_df_combined_percentages.loc[
    :, ["year", "None", "Kaggle Notebooks", "Colab Notebooks"]
]
df_cell40 = notebooks_df_combined_percentages_cell40.melt(
    id_vars=["year"], value_vars=["None", "Kaggle Notebooks", "Colab Notebooks"]
)
df_cell40 = df.rename(columns={"variable": ""})
df_cell40.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    40 non-null     object
 1           40 non-null     object
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 1.88 s, sys: 315 ms, total: 2.2 s
Wall time: 2.13 s


In [34]:
%%time
### cell 29 ###


def grab_subset_of_data_41(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell41 = (
    "Do you use any of the following data visualization libraries on a regular basis?"
)
notebooks_df_2022_cell41 = grab_subset_of_data_41(
    responses_df_2022_cell10, question_of_interest_cell41
)
notebooks_df_2022_cell41.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16326 entries, 1 to 23996
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Matplotlib                14010 non-null  object 
 1   Seaborn                   10512 non-null  object 
 2   Plotly / Plotly Express   5078 non-null   object 
 3   Ggplot / ggplot2          4145 non-null   object 
 4   Shiny                     1043 non-null   object 
 5   D3 js                     734 non-null    object 
 6   Altair                    300 non-null    object 
 7   Bokeh                     771 non-null    object 
 8   Geoplotlib                1167 non-null   object 
 9   Leaflet / Folium          554 non-null    object 
 10  Pygal                     318 non-null    object 
 11  Dygraphs                  225 non-null    object 
 12  Highcharter               198 non-null    object 
 13  None                      0 non-null      float64
 14  Other      

In [35]:
%%time
### cell 30 ###


def count_then_return_percent_42(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell42 = (
    "For how many years have you used machine learning methods?"
)
responses_in_order_cell42 = [
    "I do not use machine learning methods",
    "Under 1 year",
    "1-2 years",
    "2-3 years",
    "3-4 years",
    "4-5 years",
    "5-10 years",
    "10-20 years",
]
percentages_cell42 = count_then_return_percent_42(
    responses_df_2022_cell10, question_of_interest_cell42
).sort_index()[responses_in_order_cell42]
percentages_cell42.info()

<class 'pandas.core.series.Series'>
Index: 8 entries, I do not use machine learning methods to 10-20 years
Series name: count
Non-Null Count  Dtype  
--------------  -----  
8 non-null      float64
dtypes: float64(1)
memory usage: 128.0+ bytes
CPU times: user 71.9 ms, sys: 4.43 ms, total: 76.3 ms
Wall time: 70.9 ms


In [36]:
%%time
### cell 31 ###


def count_then_return_percent_43(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_43(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_43(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_43(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_43(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_43(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell43 = (
    "For how many years have you used machine learning methods?"
)
responses_df_2018_cell10.rename(
    columns={
        "For how many years have you used machine learning methods (at work or in school)?": question_of_interest_cell43
    },
    inplace=True,
)
responses_df_2018_cell10[question_of_interest_cell43].replace(
    ["< 1 year"], "Under 1 year", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell43].replace(
    ["10-15 years"], "10-20 years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell43].replace(
    ["20+ years"], "10-20 years", inplace=True
)
responses_df_2018_cell10[question_of_interest_cell43].replace(
    ["I have never studied machine learning but plan to learn in the future"],
    "I do not use machine learning methods",
    inplace=True,
)
responses_df_2018_cell10[question_of_interest_cell43].replace(
    ["I have never studied machine learning and I do not plan to"],
    "I do not use machine learning methods",
    inplace=True,
)
responses_df_2019_cell10[question_of_interest_cell43].replace(
    ["< 1 years"], "Under 1 year", inplace=True
)
responses_df_2019_cell10[question_of_interest_cell43].replace(
    ["10-15 years"], "10-20 years", inplace=True
)
responses_df_2019_cell10[question_of_interest_cell43].replace(
    ["20+ years"], "20 or more years", inplace=True
)
title_for_x_axis_cell43 = ""
ml_exp_df_combined = combine_subset_of_data_from_multiple_years_43(
    question_of_interest_cell43, title_for_x_axis_cell43
).sort_values(by=["year", "percentage"], ascending=True)
ml_exp_df_combined.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48 entries, 10-20 years to Under 1 year
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  48 non-null     float64
 1   year        48 non-null     object 
 2               43 non-null     object 
dtypes: float64(1), object(2)
memory usage: 1.5+ KB
CPU times: user 532 ms, sys: 29.1 ms, total: 561 ms
Wall time: 530 ms


In [37]:
%%time
### cell 32 ###


def grab_subset_of_data_44(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell44 = (
    "Which of the following machine learning frameworks do you use on a regular basis?"
)
ml_frameworks_df_2022 = grab_subset_of_data_44(
    responses_df_2022_cell10, question_of_interest_cell44
)
ml_frameworks_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14531 entries, 1 to 23996
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Scikit—learn        11403 non-null  object 
 1   TensorFlow          7953 non-null   object 
 2   Keras               6575 non-null   object 
 3   PyTorch             5191 non-null   object 
 4   Fast.ai             648 non-null    object 
 5   Xgboost             4477 non-null   object 
 6   LightGBM            1940 non-null   object 
 7   CatBoost            1165 non-null   object 
 8   Caret               821 non-null    object 
 9   Tidymodels          547 non-null    object 
 10  JAX                 252 non-null    object 
 11  PyTorch Lightning   1013 non-null   object 
 12  Huggingface         1332 non-null   object 
 13  None                0 non-null      float64
 14  Other               620 non-null    object 
dtypes: float64(1), object(14)
memory usage: 1.8+ MB
CPU times:

In [38]:
%%time
### cell 33 ###


def grab_subset_of_data_45(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


ml_frameworks_df_2022_cell45 = grab_subset_of_data_45(
    responses_df_2022_cell10, question_of_interest_cell44
)
columns_to_combine_cell45 = ["TensorFlow ", "Keras "]
ml_frameworks_df_2022_cell45["TensorFlow/Keras"] = (
    ml_frameworks_df_2022_cell45[columns_to_combine_cell45].notna().any(axis="columns")
)
ml_frameworks_df_2022_cell45 = ml_frameworks_df_2022_cell45.drop(
    columns=columns_to_combine_cell45
)
ml_frameworks_df_2022_cell45["TensorFlow/Keras"].replace(
    [True], "TensorFlow/Keras", inplace=True
)
ml_frameworks_df_2022_cell45["TensorFlow/Keras"].replace([False], np.nan, inplace=True)
columns_to_combine_cell45 = ["PyTorch ", "PyTorch Lightning ", "Fast.ai "]
ml_frameworks_df_2022_cell45["PyTorch/PyTorch Lightning/Fast.ai"] = (
    ml_frameworks_df_2022_cell45[columns_to_combine_cell45].notna().any(axis="columns")
)
ml_frameworks_df_2022_cell45 = ml_frameworks_df_2022_cell45.drop(
    columns=columns_to_combine_cell45
)
ml_frameworks_df_2022_cell45["PyTorch/PyTorch Lightning/Fast.ai"].replace(
    [True], "PyTorch/PyTorch Lightning/Fast.ai", inplace=True
)
ml_frameworks_df_2022_cell45["PyTorch/PyTorch Lightning/Fast.ai"].replace(
    [False], np.nan, inplace=True
)
columns_to_combine_cell45 = ["Xgboost ", "LightGBM ", "CatBoost "]
ml_frameworks_df_2022_cell45["Xgboost/LightGBM/CatBoost"] = (
    ml_frameworks_df_2022_cell45[columns_to_combine_cell45].notna().any(axis="columns")
)
ml_frameworks_df_2022_cell45 = ml_frameworks_df_2022_cell45.drop(
    columns=columns_to_combine_cell45
)
ml_frameworks_df_2022_cell45["Xgboost/LightGBM/CatBoost"].replace(
    [True], "Xgboost/LightGBM/CatBoost", inplace=True
)
ml_frameworks_df_2022_cell45["Xgboost/LightGBM/CatBoost"].replace(
    [False], np.nan, inplace=True
)
ml_frameworks_df_2022_cell45 = ml_frameworks_df_2022_cell45.loc[
    :,
    [
        "Scikit—learn ",
        "TensorFlow/Keras",
        "PyTorch/PyTorch Lightning/Fast.ai",
        "Xgboost/LightGBM/CatBoost",
    ],
]
ml_frameworks_df_2022_cell45.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14531 entries, 1 to 23996
Data columns (total 4 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Scikit—learn                       11403 non-null  object
 1   TensorFlow/Keras                   14531 non-null  bool  
 2   PyTorch/PyTorch Lightning/Fast.ai  14531 non-null  bool  
 3   Xgboost/LightGBM/CatBoost          14531 non-null  bool  
dtypes: bool(3), object(1)
memory usage: 269.6+ KB
CPU times: user 233 ms, sys: 16.4 ms, total: 249 ms
Wall time: 240 ms


In [39]:
%%time
### cell 34 ###


def grab_subset_of_data_46(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_46(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_46(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_46(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_46(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_46(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_46(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_46(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_46(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_46(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_46(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_46(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_46(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_46(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_46(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_46(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_old = (
    "What machine learning frameworks have you used in the past 5 years?"
)
question_of_interest_new_cell46 = (
    "Which of the following machine learning frameworks do you use on a regular basis?"
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    question_of_interest_old, question_of_interest_new_cell46, regex=False
)
question_of_interest_cell46 = (
    "Which of the following machine learning frameworks do you use on a regular basis?"
)
(ml_df_combined, ml_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_46(
        question_of_interest_cell46
    )
)
columns_to_combine_cell46 = ["TensorFlow", "TensorFlow "]
ml_df_combined["TensorFlow_"] = (
    ml_df_combined[columns_to_combine_cell46].notna().any(axis="columns")
)
ml_df_combined_cell46 = ml_df_combined.drop(columns=columns_to_combine_cell46)
ml_df_combined_cell46["TensorFlow_"].replace([True], "TensorFlow", inplace=True)
ml_df_combined_cell46["TensorFlow_"].replace([False], np.nan, inplace=True)
ml_df_combined_cell46.columns = ml_df_combined_cell46.columns.str.replace(
    "TensorFlow_", "TensorFlow ", regex=False
)
columns_to_combine_cell46 = ["Keras", "Keras "]
ml_df_combined_cell46["Keras_"] = (
    ml_df_combined_cell46[columns_to_combine_cell46].notna().any(axis="columns")
)
ml_df_combined_cell46 = ml_df_combined_cell46.drop(columns=columns_to_combine_cell46)
ml_df_combined_cell46["Keras_"].replace([True], "Keras", inplace=True)
ml_df_combined_cell46["Keras_"].replace([False], np.nan, inplace=True)
ml_df_combined_cell46.columns = ml_df_combined_cell46.columns.str.replace(
    "Keras_", "Keras ", regex=False
)
columns_to_combine_cell46 = ["PyTorch", "PyTorch "]
ml_df_combined_cell46["PyTorch_"] = (
    ml_df_combined_cell46[columns_to_combine_cell46].notna().any(axis="columns")
)
ml_df_combined_cell46 = ml_df_combined_cell46.drop(columns=columns_to_combine_cell46)
ml_df_combined_cell46["PyTorch_"].replace([True], "PyTorch", inplace=True)
ml_df_combined_cell46["PyTorch_"].replace([False], np.nan, inplace=True)
ml_df_combined_cell46.columns = ml_df_combined_cell46.columns.str.replace(
    "PyTorch_", "PyTorch ", regex=False
)
columns_to_combine_cell46 = ["Scikit—learn ", "Learn", "learn "]
ml_df_combined_cell46["Scikit—learn_"] = (
    ml_df_combined_cell46[columns_to_combine_cell46].notna().any(axis="columns")
)
ml_df_combined_cell46 = ml_df_combined_cell46.drop(columns=columns_to_combine_cell46)
ml_df_combined_cell46["Scikit—learn_"].replace([True], "Scikit—learn", inplace=True)
ml_df_combined_cell46["Scikit—learn_"].replace([False], np.nan, inplace=True)
ml_df_combined_cell46.columns = ml_df_combined_cell46.columns.str.replace(
    "Scikit—learn_", "Scikit-learn ", regex=False
)
ml_df_combined_counts_2 = ml_df_combined_cell46.groupby("year").count().reset_index()
ml_df_combined_percentages = convert_df_of_counts_to_percentages_46(
    ml_df_combined_cell46, ml_df_combined_counts_2
)
ml_df_combined_percentages_cell46 = ml_df_combined_percentages.loc[
    :, ["year", "Scikit-learn ", "TensorFlow ", "Keras ", "PyTorch ", "None", "Other"]
]
df_cell46 = ml_df_combined_percentages_cell46.melt(
    id_vars=["year"],
    value_vars=["Scikit-learn ", "TensorFlow ", "Keras ", "PyTorch ", "None", "Other"],
)
df_cell46 = df.sort_values(by=["year", "value"], ascending=True)
df_cell46 = df.rename(columns={"variable": ""})
df_cell46.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    40 non-null     object
 1           40 non-null     object
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 1.71 s, sys: 311 ms, total: 2.03 s
Wall time: 1.99 s


In [40]:
%%time
### cell 35 ###


def grab_subset_of_data_47(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_47(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_47(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_47(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_47(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_47(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_47(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_47(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_47(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_47(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_47(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_47(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_47(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_47(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_47(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_47(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_old_cell47 = (
    "What machine learning frameworks have you used in the past 5 years?"
)
question_of_interest_new_cell47 = (
    "Which of the following machine learning frameworks do you use on a regular basis?"
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    question_of_interest_old_cell47, question_of_interest_new_cell47, regex=False
)
question_of_interest_cell47 = (
    "Which of the following machine learning frameworks do you use on a regular basis?"
)
(ml_df_combined_cell47, ml_df_combined_counts_cell47) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
        question_of_interest_cell47
    )
)
ml_df_combined_2 = ml_df_combined_cell47.copy()
columns_to_combine_cell47 = ["TensorFlow", "TensorFlow ", "Keras", "Keras "]
ml_df_combined_2["TensorFlow/Keras"] = (
    ml_df_combined_2[columns_to_combine_cell47].notna().any(axis="columns")
)
ml_df_combined_2_cell47 = ml_df_combined_2.drop(columns=columns_to_combine_cell47)
ml_df_combined_2_cell47["TensorFlow/Keras"].replace(
    [True], "TensorFlow/Keras", inplace=True
)
ml_df_combined_2_cell47["TensorFlow/Keras"].replace([False], np.nan, inplace=True)
columns_to_combine_cell47 = [
    "PyTorch",
    "PyTorch ",
    "PyTorch Lightning ",
    "Fast.ai ",
    "Fastai",
]
ml_df_combined_2_cell47["PyTorch/Lightning/Fast.ai"] = (
    ml_df_combined_2_cell47[columns_to_combine_cell47].notna().any(axis="columns")
)
ml_df_combined_2_cell47 = ml_df_combined_2_cell47.drop(
    columns=columns_to_combine_cell47
)
ml_df_combined_2_cell47["PyTorch/Lightning/Fast.ai"].replace(
    [True], "PyTorch/PyTorch Lightning/Fast.ai", inplace=True
)
ml_df_combined_2_cell47["PyTorch/Lightning/Fast.ai"].replace(
    [False], np.nan, inplace=True
)
columns_to_combine_cell47 = [
    "Xgboost",
    "Xgboost ",
    "lightgbm",
    "LightGBM ",
    "catboost",
    "CatBoost ",
]
ml_df_combined_2_cell47["Xgboost/LightGBM/CatBoost"] = (
    ml_df_combined_2_cell47[columns_to_combine_cell47].notna().any(axis="columns")
)
ml_df_combined_2_cell47 = ml_df_combined_2_cell47.drop(
    columns=columns_to_combine_cell47
)
ml_df_combined_2_cell47["Xgboost/LightGBM/CatBoost"].replace(
    [True], "Xgboost/LightGBM/CatBoost", inplace=True
)
ml_df_combined_2_cell47["Xgboost/LightGBM/CatBoost"].replace(
    [False], np.nan, inplace=True
)
columns_to_combine_cell47 = ["Scikit—learn ", "Scikit—learn ", "learn ", "Learn"]
ml_df_combined_2_cell47["Scikit-learn_"] = (
    ml_df_combined_2_cell47[columns_to_combine_cell47].notna().any(axis="columns")
)
ml_df_combined_2_cell47 = ml_df_combined_2_cell47.drop(
    columns=columns_to_combine_cell47
)
ml_df_combined_2_cell47["Scikit-learn_"].replace([True], "Scikit-learn_", inplace=True)
ml_df_combined_2_cell47["Scikit-learn_"].replace([False], np.nan, inplace=True)
ml_df_combined_2_cell47.columns = ml_df_combined_2_cell47.columns.str.replace(
    "Scikit-learn_", "Scikit-learn", regex=False
)
ml_df_combined_counts_2_cell47 = (
    ml_df_combined_2_cell47.groupby("year").count().reset_index()
)
ml_df_combined_percentages_cell47 = convert_df_of_counts_to_percentages_47(
    ml_df_combined_2_cell47, ml_df_combined_counts_2_cell47
)
ml_df_combined_percentages_cell47 = ml_df_combined_percentages_cell47.loc[
    :,
    [
        "year",
        "Scikit-learn",
        "TensorFlow/Keras",
        "PyTorch/Lightning/Fast.ai",
        "Xgboost/LightGBM/CatBoost",
    ],
]
df_cell47 = ml_df_combined_percentages_cell47.melt(
    id_vars=["year"],
    value_vars=[
        "Scikit-learn",
        "Xgboost/LightGBM/CatBoost",
        "TensorFlow/Keras",
        "PyTorch/Lightning/Fast.ai",
    ],
)
df_cell47 = df.sort_values(by=["year", "value"], ascending=True)
df_cell47 = df.rename(columns={"variable": ""})
df_cell47.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    40 non-null     object
 1           40 non-null     object
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 1.65 s, sys: 340 ms, total: 1.99 s
Wall time: 1.95 s


In [41]:
%%time
### cell 36 ###


def grab_subset_of_data_48(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell48 = (
    "Which of the following ML algorithms do you use on a regular basis?"
)
ml_algos_df_2022 = grab_subset_of_data_48(
    responses_df_2022_cell10, question_of_interest_cell48
)
ml_algos_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14538 entries, 1 to 23996
Data columns (total 14 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   Linear or Logistic Regression                        11338 non-null  object 
 1   Decision Trees or Random Forests                     9373 non-null   object 
 2   Gradient Boosting Machines (xgboost, lightgbm, etc)  5506 non-null   object 
 3   Bayesian Approaches                                  3661 non-null   object 
 4   Evolutionary Approaches                              823 non-null    object 
 5   Dense Neural Networks (MLPs, etc)                    3476 non-null   object 
 6   Convolutional Neural Networks                        6006 non-null   object 
 7   Generative Adversarial Networks                      1166 non-null   object 
 8   Recurrent Neural Networks                            3451 non-null   ob

In [42]:
%%time
### cell 37 ###


def grab_subset_of_data_49(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell49 = (
    "Which categories of computer vision methods do you use on a regular basis?"
)
cv_df_2022 = grab_subset_of_data_49(
    responses_df_2022_cell10, question_of_interest_cell49
)
cv_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5263 entries, 7 to 23992
Data columns (total 8 columns):
 #   Column                                                                                                                Non-Null Count  Dtype  
---  ------                                                                                                                --------------  -----  
 0   General purpose image/video tools (PIL, cv2, skimage, etc)                                                            2293 non-null   object 
 1   Image segmentation methods (U—Net, Mask R—CNN, etc)                                                                   2495 non-null   object 
 2   Object detection methods (YOLOv6, RetinaNet, etc)                                                                     2525 non-null   object 
 3   Image classification and other general purpose networks (VGG, Inception, ResNet, ResNeXt, NASNet, EfficientNet, etc)  3664 non-null   object 
 4   Generative 

In [43]:
%%time
### cell 38 ###


def grab_subset_of_data_50(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell50 = "Which of the following natural language processing (NLP) methods do you use on a regular basis?"
nlp_df_2022 = grab_subset_of_data_50(
    responses_df_2022_cell10, question_of_interest_cell50
)
nlp_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3319 entries, 3 to 23983
Data columns (total 6 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   Word embeddings/vectors (GLoVe, fastText, word2vec)     2241 non-null   object 
 1   Encoder—decoder models (seq2seq, vanilla transformers)  1701 non-null   object 
 2   Contextualized embeddings (ELMo, CoVe)                  685 non-null    object 
 3   Transformer language models (GPT—3, BERT, XLnet, etc)   2219 non-null   object 
 4   None                                                    0 non-null      float64
 5   Other                                                   126 non-null    object 
dtypes: float64(1), object(5)
memory usage: 181.5+ KB
CPU times: user 129 ms, sys: 32.7 ms, total: 162 ms
Wall time: 155 ms


In [44]:
%%time
### cell 39 ###


def grab_subset_of_data_51(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell51 = "Which of the following natural language processing (NLP) methods do you use on a regular basis?"
nlp_df_2022_cell51 = grab_subset_of_data_51(
    responses_df_2022_cell10, question_of_interest_cell51
)
counts_2022 = (
    nlp_df_2022_cell51[nlp_df_2022_cell51.columns[:]]
    .count()
    .sort_values(ascending=True)
)
percentages_2022 = counts_2022 * 100 / len(nlp_df_2022_cell51)
transformers = "Transformer language models (GPT—3, BERT, XLnet, etc)"
percentages_2022_cell51 = percentages_2022.rename({transformers: "Transformers"})[
    ["Transformers"]
]
nlp_df_2021 = grab_subset_of_data_51(responses_df_2021, question_of_interest_cell51)
counts_2021 = nlp_df_2021[nlp_df_2021.columns[:]].count().sort_values(ascending=True)
percentages_2021 = counts_2021 * 100 / len(nlp_df_2021)
transformers_cell51 = "3, BERT, XLnet, etc)"
percentages_2021_cell51 = percentages_2021.rename(
    {transformers_cell51: "Transformers"}
)[["Transformers"]]
nlp_df_2020 = grab_subset_of_data_51(responses_df_2020, question_of_interest_cell51)
counts_2020 = nlp_df_2020[nlp_df_2020.columns[:]].count().sort_values(ascending=True)
percentages_2020 = counts_2020 * 100 / len(nlp_df_2020)
transformers_cell51 = "3, BERT, XLnet, etc)"
percentages_2020_cell51 = percentages_2020.rename(
    {transformers_cell51: "Transformers"}
)[["Transformers"]]
nlp_df_2019 = grab_subset_of_data_51(
    responses_df_2019_cell10, question_of_interest_cell51
)
counts_2019 = nlp_df_2019[nlp_df_2019.columns[:]].count().sort_values(ascending=True)
percentages_2019 = counts_2019 * 100 / len(nlp_df_2019)
transformers_cell51 = "2, BERT, XLnet, etc)"
percentages_2019_cell51 = percentages_2019.rename(
    {transformers_cell51: "Transformers"}
)[["Transformers"]]
(
    percentages_2019_cell51.info(),
    percentages_2020_cell51.info(),
    percentages_2021_cell51.info(),
    percentages_2022_cell51.info(),
)

<class 'pandas.core.series.Series'>
Index: 1 entries, Transformers to Transformers
Series name: None
Non-Null Count  Dtype  
--------------  -----  
1 non-null      float64
dtypes: float64(1)
memory usage: 16.0+ bytes
<class 'pandas.core.series.Series'>
Index: 1 entries, Transformers to Transformers
Series name: None
Non-Null Count  Dtype  
--------------  -----  
1 non-null      float64
dtypes: float64(1)
memory usage: 16.0+ bytes
<class 'pandas.core.series.Series'>
Index: 1 entries, Transformers to Transformers
Series name: None
Non-Null Count  Dtype  
--------------  -----  
1 non-null      float64
dtypes: float64(1)
memory usage: 16.0+ bytes
<class 'pandas.core.series.Series'>
Index: 1 entries, Transformers to Transformers
Series name: None
Non-Null Count  Dtype  
--------------  -----  
1 non-null      float64
dtypes: float64(1)
memory usage: 16.0+ bytes
CPU times: user 700 ms, sys: 134 ms, total: 834 ms
Wall time: 802 ms


(None, None, None, None)

In [45]:
%%time
### cell 40 ###


def grab_subset_of_data_52(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell52 = (
    "Do you download pre—trained model weights from any of the following services?"
)
models_df_2022 = grab_subset_of_data_52(
    responses_df_2022_cell10, question_of_interest_cell52
)
models_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15420 entries, 1 to 23996
Data columns (total 10 columns):
 #   Column                                                              Non-Null Count  Dtype 
---  ------                                                              --------------  ----- 
 0   TensorFlow Hub                                                      3087 non-null   object
 1   PyTorch Hub                                                         2012 non-null   object
 2   Huggingface Models                                                  1793 non-null   object
 3   Timm                                                                434 non-null    object
 4   Jumpstart                                                           237 non-null    object
 5   ONNX models                                                         418 non-null    object
 6   NVIDIA NGC models                                                   630 non-null    object
 7   Kaggle datasets            

In [46]:
%%time
### cell 41 ###


def bar_chart_multiple_choice_53(
    response_counts, title, y_axis_title, orientation="h", num_choices=15
):
    response_counts_series = pd.Series(response_counts)
    response_counts_series.index.name = ""
    pd.DataFrame(response_counts_series).to_csv(
        "/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/"
        + title
        + ".csv",
        index=True,
    )
    tmp_cmp = max(response_counts_series[0:num_choices] * 1.2)


def count_then_return_percent_53(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell53 = "Which of the following ML model hubs/repositories do you use most often? - Selected Choice"
responses_in_order_cell53 = [
    " Kaggle datasets ",
    " Huggingface Models ",
    "  TensorFlow Hub ",
    " PyTorch Hub ",
    " Timm ",
    " NVIDIA NGC models  ",
    " ONNX models ",
    " Jumpstart ",
]
percentages_cell53 = (
    count_then_return_percent_53(responses_df_2022_cell10, question_of_interest_cell53)
    .sort_index()[responses_in_order_cell53]
    .iloc[::-1]
)
title_for_chart_cell53 = (
    "Favorite ML model repository (for users of multiple repositories)"
)
title_for_y_axis_cell53 = "% of respondents"
orientation_for_chart_cell53 = "h"
bar_chart_multiple_choice_53(
    response_counts=percentages_cell53,
    title=title_for_chart_cell53,
    y_axis_title=title_for_y_axis_cell53,
    orientation=orientation_for_chart_cell53,
    num_choices=9,
)

CPU times: user 84 ms, sys: 3.35 ms, total: 87.4 ms
Wall time: 79.9 ms


In [47]:
%%time
### cell 42 ###


def count_then_return_percent_54(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell54 = "Select the title most similar to your current role (or most recent title if retired): - Selected Choice"
percentages_cell54 = count_then_return_percent_54(
    responses_df_2022_cell10, question_of_interest_cell54
).sort_index()
percentages_cell54 = percentages.sort_values(ascending=True)
percentages_cell54.info()

<class 'pandas.core.series.Series'>
Index: 13 entries, Turkey to India
Series name: count
Non-Null Count  Dtype  
--------------  -----  
13 non-null     float64
dtypes: float64(1)
memory usage: 208.0+ bytes
CPU times: user 69.3 ms, sys: 3.85 ms, total: 73.1 ms
Wall time: 68.9 ms


In [48]:
%%time
### cell 43 ###


def count_then_return_percent_55(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_55(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_55(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_55(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_55(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_55(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell55 = "Select the title most similar to your current role (or most recent title if retired): - Selected Choice"
job_df_combined = combine_subset_of_data_from_multiple_years_55(
    question_of_interest_cell55, title_for_x_axis_cell43
)
job_df_combined.replace(
    ["Data Analyst (Business, Marketing, Financial, Quantitative, etc)"],
    "Data Analyst",
    inplace=True,
)
job_df_combined_cell55 = job_df_combined.rename(
    {"Data Analyst (Business, Marketing, Financial, Quantitative, etc)": "Data Analyst"}
)
job_df_combined_cell55 = job_df_combined[
    job_df_combined.index.isin(
        [
            "Data Analyst",
            "Data Engineer",
            "Data Scientist",
            "Research Scientist",
            "Software Engineer",
        ]
    )
]
job_df_combined_cell55.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 24 entries, Data Analyst to Software Engineer
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   percentage  24 non-null     float64
 1   year        24 non-null     object
 2               24 non-null     object
dtypes: float64(1), object(2)
memory usage: 1.3+ KB
CPU times: user 374 ms, sys: 11.3 ms, total: 385 ms
Wall time: 364 ms


In [49]:
%%time
### cell 44 ###


def count_then_return_percent_56(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell56 = "In what industry is your current employer/contract (or your most recent employer if retired)? - Selected Choice"
percentages_cell56 = count_then_return_percent_56(
    responses_df_2022_cell10, question_of_interest_cell56
).sort_index()
percentages_cell56 = percentages.sort_values(ascending=True)
percentages_cell56.info()

<class 'pandas.core.series.Series'>
Index: 13 entries, Turkey to India
Series name: count
Non-Null Count  Dtype  
--------------  -----  
13 non-null     float64
dtypes: float64(1)
memory usage: 208.0+ bytes
CPU times: user 72.8 ms, sys: 4.03 ms, total: 76.8 ms
Wall time: 71.8 ms


In [50]:
%%time
### cell 45 ###


def count_then_return_percent_57(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell57 = "What is the size of the company where you are employed?"
responses_in_order_cell57 = [
    "0-49 employees",
    "50-249 employees",
    "250-999 employees",
    "1000-9,999 employees",
    "10,000 or more employees",
]
percentages_cell57 = count_then_return_percent_57(
    responses_df_2022_cell10, question_of_interest_cell57
).sort_index()[responses_in_order_cell57]
percentages_cell57.info()

<class 'pandas.core.series.Series'>
Index: 5 entries, 0-49 employees to 10,000 or more employees
Series name: count
Non-Null Count  Dtype  
--------------  -----  
5 non-null      float64
dtypes: float64(1)
memory usage: 80.0+ bytes
CPU times: user 77.3 ms, sys: 3.89 ms, total: 81.2 ms
Wall time: 75.7 ms


In [51]:
%%time
### cell 46 ###


def count_then_return_percent_58(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell58 = "Approximately how many individuals are responsible for data science workloads at your place of business?"
responses_in_order_cell58 = ["0", "1-2", "3-4", "5-9", "10-14", "15-19", "20+"]
percentages_cell58 = count_then_return_percent_58(
    responses_df_2022_cell10, question_of_interest_cell58
).sort_index()[responses_in_order_cell58]
percentages_cell58.info()

<class 'pandas.core.series.Series'>
Index: 7 entries, 0 to 20+
Series name: count
Non-Null Count  Dtype  
--------------  -----  
7 non-null      float64
dtypes: float64(1)
memory usage: 112.0+ bytes
CPU times: user 101 ms, sys: 3.49 ms, total: 105 ms
Wall time: 100 ms


In [52]:
%%time
### cell 47 ###


def count_then_return_percent_59(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell59 = "Does your current employer incorporate machine learning methods into their business?"
responses_in_order_cell59 = [
    "No (we do not use ML methods)",
    "I do not know",
    "We are exploring ML methods (and may one day put a model into production)",
    "We recently started using ML methods (i.e., models in production for less than 2 years)",
    "We have well established ML methods (i.e., models in production for more than 2 years)",
]
percentages_cell59 = count_then_return_percent_59(
    responses_df_2022_cell10, question_of_interest_cell59
).sort_index()[responses_in_order_cell59]
percentages_cell59.info()

<class 'pandas.core.series.Series'>
Index: 5 entries, No (we do not use ML methods) to We have well established ML methods (i.e., models in production for more than 2 years)
Series name: count
Non-Null Count  Dtype  
--------------  -----  
5 non-null      float64
dtypes: float64(1)
memory usage: 80.0+ bytes
CPU times: user 74.9 ms, sys: 253 μs, total: 75.2 ms
Wall time: 71.5 ms


In [53]:
%%time
### cell 48 ###


def count_then_return_percent_60(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


def add_year_column_to_dataframes_60(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def combine_subset_of_data_from_multiple_years_60(
    question_of_interest, x_axis_title, include_2017=None
):
    if include_2017 is not None:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        df_2017 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2017, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_60(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_2017.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined
    else:
        question_of_interest = question_of_interest
        df_2022 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2022_cell10, question_of_interest
            ).sort_index()
        )
        df_2021 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2021, question_of_interest
            ).sort_index()
        )
        df_2020 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2020, question_of_interest
            ).sort_index()
        )
        df_2019 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2019_cell10, question_of_interest
            ).sort_index()
        )
        df_2018 = pd.DataFrame(
            count_then_return_percent_60(
                responses_df_2018_cell10, question_of_interest
            ).sort_index()
        )
        add_year_column_to_dataframes_60(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_2022.columns = ["percentage", "year"]
        df_2021.columns = ["percentage", "year"]
        df_2020.columns = ["percentage", "year"]
        df_2019.columns = ["percentage", "year"]
        df_2018.columns = ["percentage", "year"]
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined[x_axis_title] = df_combined.index
        df_combined.columns = ["percentage", "year", x_axis_title]
        return df_combined


question_of_interest_cell60 = "Does your current employer incorporate machine learning methods into their business?"
title_for_x_axis_cell60 = ""
maturity_df_combined = combine_subset_of_data_from_multiple_years_60(
    question_of_interest_cell60, title_for_x_axis_cell60
).sort_values(by=["year", "percentage"], ascending=True)
maturity_df_combined.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35 entries, We use ML methods for generating insights (but do not put working models into production) to nan
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  35 non-null     float64
 1   year        35 non-null     object 
 2               30 non-null     object 
dtypes: float64(1), object(2)
memory usage: 1.1+ KB
CPU times: user 330 ms, sys: 16.5 ms, total: 347 ms
Wall time: 329 ms


In [54]:
%%time
### cell 49 ###


def grab_subset_of_data_61(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell61 = (
    "Select any activities that make up an important part of your role at work"
)
models_df_2022_cell61 = grab_subset_of_data_61(
    responses_df_2022_cell10, question_of_interest_cell61
)
models_df_2022_cell61.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8652 entries, 3 to 23995
Data columns (total 8 columns):
 #   Column                                                                                                            Non-Null Count  Dtype 
---  ------                                                                                                            --------------  ----- 
 0   Analyze and understand data to influence product or business decisions                                            4999 non-null   object
 1   Build and/or run the data infrastructure that my business uses for storing, analyzing, and operationalizing data  2622 non-null   object
 2   Build prototypes to explore applying machine learning to new areas                                                3034 non-null   object
 3   Build and/or run a machine learning service that operationally improves my product or workflows                   2171 non-null   object
 4   Experimentation and iteration to improve 

In [55]:
%%time
# accepted
### cell 50 ###


def count_then_return_percent_62(dataframe, x_axis_title, order):
    # Compute normalized counts (percentages) on GPU
    percentages = (
        dataframe[x_axis_title].value_counts(dropna=False, normalize=True) * 100
    ).round(1)
    # Reindex to enforce the desired order and fill any missing categories with 0
    return percentages.reindex(order).fillna(0)


question_of_interest_cell62 = (
    "What is your current yearly compensation (approximate $USD)?"
)
responses_in_order_cell62 = [
    "$0-999",
    "1,000-1,999",
    "2,000-2,999",
    "3,000-3,999",
    "4,000-4,999",
    "5,000-7,499",
    "7,500-9,999",
    "10,000-14,999",
    "15,000-19,999",
    "20,000-24,999",
    "25,000-29,999",
    "30,000-39,999",
    "40,000-49,999",
    "50,000-59,999",
    "60,000-69,999",
    "70,000-79,999",
    "80,000-89,999",
    "90,000-99,999",
    "100,000-124,999",
    "125,000-149,999",
    "150,000-199,999",
    "200,000-249,999",
    "250,000-299,999",
    "300,000-499,999",
    "$500,000-999,999",
    ">$1,000,000",
]

percentages_cell62 = count_then_return_percent_62(
    responses_df_2022_cell10, question_of_interest_cell62, responses_in_order_cell62
)
percentages_cell62.info()

<class 'pandas.core.series.Series'>
Index: 26 entries, $0-999 to >$1,000,000
Series name: proportion
Non-Null Count  Dtype  
--------------  -----  
26 non-null     float64
dtypes: float64(1)
memory usage: 416.0+ bytes
CPU times: user 41.6 ms, sys: 3.47 ms, total: 45.1 ms
Wall time: 42.1 ms


In [56]:
%%time
### cell 51 ###


def count_then_return_percent_63(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_cell63 = (
    "What is your current yearly compensation (approximate $USD)?"
)
USA_responses_df_2022_cell63 = responses_df_2022_cell10[
    responses_df_2022_cell10["In which country do you currently reside?"]
    == "United States of America"
]
responses_in_order_cell63 = [
    "$0-999",
    "1,000-1,999",
    "2,000-2,999",
    "3,000-3,999",
    "5,000-7,499",
    "7,500-9,999",
    "10,000-14,999",
    "15,000-19,999",
    "20,000-24,999",
    "25,000-29,999",
    "30,000-39,999",
    "40,000-49,999",
    "50,000-59,999",
    "60,000-69,999",
    "70,000-79,999",
    "80,000-89,999",
    "90,000-99,999",
    "100,000-124,999",
    "125,000-149,999",
    "150,000-199,999",
    "200,000-249,999",
    "250,000-299,999",
    "300,000-499,999",
    "$500,000-999,999",
    ">$1,000,000",
]
percentages_cell63 = count_then_return_percent_63(
    USA_responses_df_2022_cell63, question_of_interest_cell63
).sort_index()[responses_in_order_cell63]
percentages_cell63.info()

<class 'pandas.core.series.Series'>
Index: 25 entries, $0-999 to >$1,000,000
Series name: count
Non-Null Count  Dtype  
--------------  -----  
25 non-null     float64
dtypes: float64(1)
memory usage: 400.0+ bytes
CPU times: user 103 ms, sys: 4.11 ms, total: 107 ms
Wall time: 102 ms


In [57]:
%%time
# accepted
### cell 52 ###

question_of_interest_cell64 = (
    "What is your current yearly compensation (approximate $USD)?"
)
# create a boolean mask for respondents in India
mask = responses_df_2022_cell10["In which country do you currently reside?"] == "India"
# define the exact order of salary bins
responses_in_order_cell64 = [
    "$0-999",
    "1,000-1,999",
    "2,000-2,999",
    "3,000-3,999",
    "4,000-4,999",
    "5,000-7,499",
    "7,500-9,999",
    "10,000-14,999",
    "15,000-19,999",
    "20,000-24,999",
    "25,000-29,999",
    "30,000-39,999",
    "40,000-49,999",
    "50,000-59,999",
    "60,000-69,999",
    "70,000-79,999",
    "80,000-89,999",
    "90,000-99,999",
    "100,000-124,999",
    "125,000-149,999",
    "150,000-199,999",
    "200,000-249,999",
    "250,000-299,999",
    "300,000-499,999",
    "$500,000-999,999",
    ">$1,000,000",
]

# compute normalized counts (percentages), round to one decimal, and align to desired order
percentages_cell64 = (
    responses_df_2022_cell10.loc[mask, question_of_interest_cell64]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(1)
    .reindex(responses_in_order_cell64)
)
# match the original behavior by printing the Series info (returns None)
percentages_cell64.info()

<class 'pandas.core.series.Series'>
Index: 26 entries, $0-999 to >$1,000,000
Series name: proportion
Non-Null Count  Dtype  
--------------  -----  
26 non-null     float64
dtypes: float64(1)
memory usage: 416.0+ bytes
CPU times: user 106 ms, sys: 4.4 ms, total: 110 ms
Wall time: 104 ms


In [58]:
%%time
### cell 53 ###


def count_then_return_percent_65(dataframe, x_axis_title):
    s = dataframe[x_axis_title]
    # compute normalized value counts on GPU, multiply by 100 and round
    percentages = s.value_counts(dropna=False, normalize=True).mul(100).round(1)
    return percentages


responses_in_order_cell65 = [
    "$0 ($USD)",
    "$1-$99",
    "$100-$999",
    "$1000-$9,999",
    "$10,000-$99,999",
    "$100,000 or more ($USD)",
]
question_of_interest_cell65 = "Approximately how much money have you spent on machine learning and/or cloud computing services at home or at work in the past 5 years (approximate $USD)?\n (approximate $USD)?"
# reindex to enforce desired order without sorting or additional indexing
percentages_cell65 = count_then_return_percent_65(
    responses_df_2022_cell10, question_of_interest_cell65
).reindex(responses_in_order_cell65)
percentages_cell65.info()

<class 'pandas.core.series.Series'>
Index: 6 entries, $0 ($USD) to $100,000 or more ($USD)
Series name: proportion
Non-Null Count  Dtype  
--------------  -----  
6 non-null      float64
dtypes: float64(1)
memory usage: 96.0+ bytes
CPU times: user 42.7 ms, sys: 271 μs, total: 42.9 ms
Wall time: 39 ms


In [59]:
%%time
### cell 54 ###


def grab_subset_of_data_66(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell66 = "Which of the following cloud computing platforms do you use? (Select all that apply)"
models_df_2022_cell66 = grab_subset_of_data_66(
    responses_df_2022_cell10, question_of_interest_cell66
)
models_df_2022_cell66.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4100 entries, 3 to 23994
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Amazon Web Services (AWS)     2346 non-null   object 
 1   Microsoft Azure               1416 non-null   object 
 2   Google Cloud Platform (GCP)   2056 non-null   object 
 3   IBM Cloud / Red Hat           287 non-null    object 
 4   Oracle Cloud                  230 non-null    object 
 5   SAP Cloud                     107 non-null    object 
 6   VMware Cloud                  155 non-null    object 
 7   Alibaba Cloud                 76 non-null     object 
 8   Tencent Cloud                 56 non-null     object 
 9   Huawei Cloud                  47 non-null     object 
 10  None                          0 non-null      float64
 11  Other                         217 non-null    object 
dtypes: float64(1), object(11)
memory usage: 416.4+ KB
CPU times: user 

In [60]:
%%time
### cell 55 ###


def grab_subset_of_data_67(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


def add_year_column_to_dataframes_67(
    df_2022, df_2021, df_2020, df_2019, df_2018, df_2017=None
):
    if df_2017 is not None:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        df_2017["year"] = "2017"
        return (df_2017, df_2018, df_2019, df_2020, df_2021, df_2022)
    else:
        df_2022["year"] = "2022"
        df_2021["year"] = "2021"
        df_2020["year"] = "2020"
        df_2019["year"] = "2019"
        df_2018["year"] = "2018"
        return (df_2018, df_2019, df_2020, df_2021, df_2022)


def convert_df_of_counts_to_percentages_67(df, df_counts):
    df_combined_percentages = df_counts.copy().astype(int)
    df_combined_percentages[0:1] = (
        df_combined_percentages[0:1] * 100 / df["year"].eq("2018").sum()
    )
    df_combined_percentages[1:2] = (
        df_combined_percentages[1:2] * 100 / df["year"].eq("2019").sum()
    )
    df_combined_percentages[2:3] = (
        df_combined_percentages[2:3] * 100 / df["year"].eq("2020").sum()
    )
    df_combined_percentages[3:4] = (
        df_combined_percentages[3:4] * 100 / df["year"].eq("2021").sum()
    )
    df_combined_percentages[4:5] = (
        df_combined_percentages[4:5] * 100 / df["year"].eq("2022").sum()
    )
    df_combined_percentages["year"] = ["2018", "2019", "2020", "2021", "2022"]
    return df_combined_percentages


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(
    question_of_interest, include_2017=None
):
    if include_2017 is None:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_67(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_67(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_67(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_67(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_67(responses_df_2018_cell10, question_of_interest)
        add_year_column_to_dataframes_67(df_2022, df_2021, df_2020, df_2019, df_2018)
        df_combined = pd.concat([df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)
    else:
        question_of_interest = question_of_interest
        df_2022 = grab_subset_of_data_67(responses_df_2022_cell10, question_of_interest)
        df_2021 = grab_subset_of_data_67(responses_df_2021, question_of_interest)
        df_2020 = grab_subset_of_data_67(responses_df_2020, question_of_interest)
        df_2019 = grab_subset_of_data_67(responses_df_2019_cell10, question_of_interest)
        df_2018 = grab_subset_of_data_67(responses_df_2018_cell10, question_of_interest)
        df_2017 = grab_subset_of_data_67(responses_df_2017, question_of_interest)
        add_year_column_to_dataframes_67(
            df_2022, df_2021, df_2020, df_2019, df_2018, df_2017
        )
        df_combined = pd.concat([df_2017, df_2018, df_2019, df_2020, df_2021, df_2022])
        df_combined_counts = df_combined.groupby("year").count().reset_index()
        return (df_combined, df_combined_counts)


question_of_interest_cell67 = "Which of the following cloud computing platforms do you use? (Select all that apply)"
question_of_interest_old_cell67 = (
    "Which of the following cloud computing platforms do you use on a regular basis?"
)
question_of_interest_old_2 = "Which of the following cloud computing services have you used at work or school in the last 5 years?"
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Amazon Web Services (AWS)", "Amazon Web Services (AWS) ", regex=False
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Google Cloud Platform (GCP)", "Google Cloud Platform (GCP) ", regex=False
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    "Microsoft Azure", "Microsoft Azure ", regex=False
)
responses_df_2021.columns = responses_df_2021.columns.str.replace(
    question_of_interest_old_cell67, question_of_interest_cell67, regex=False
)
responses_df_2020.columns = responses_df_2020.columns.str.replace(
    question_of_interest_old_cell67, question_of_interest_cell67, regex=False
)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    question_of_interest_old_cell67, question_of_interest_cell67, regex=False
)
responses_df_2018_cell10.columns = responses_df_2018_cell10.columns.str.replace(
    question_of_interest_old_2, question_of_interest_cell67, regex=False
)
(cloud_df_combined, cloud_df_combined_counts) = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_67(
        question_of_interest_cell67
    )
)
cloud_df_combined_percentages = convert_df_of_counts_to_percentages_67(
    cloud_df_combined, cloud_df_combined_counts
)
cloud_df_combined_percentages_cell67 = cloud_df_combined_percentages.loc[
    :,
    [
        "year",
        "Amazon Web Services (AWS) ",
        "Google Cloud Platform (GCP) ",
        "Microsoft Azure ",
    ],
]
df_cell67 = cloud_df_combined_percentages_cell67.melt(
    id_vars=["year"],
    value_vars=[
        "Amazon Web Services (AWS) ",
        "Google Cloud Platform (GCP) ",
        "Microsoft Azure ",
    ],
)
df_cell67 = df.rename(columns={"variable": ""})
df_cell67.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   year    40 non-null     object
 1           40 non-null     object
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 1.42 s, sys: 321 ms, total: 1.74 s
Wall time: 1.69 s


In [61]:
%%time
# accepted
### cell 56 ###


def count_then_return_percent_68(df, col):
    # pull the series once, compute raw counts, then build percentages in one chain
    s = df[col]
    counts = s.value_counts(dropna=False)
    total = s.count()
    # do the multiplication, division, and rounding on the GPU
    return (counts * 100 / total).round(1)


question_of_interest_cell68 = (
    "Of the cloud platforms that you are familiar with, "
    "which has the best developer experience (most enjoyable to use)? - Selected Choice"
)
responses_in_order_cell68 = [
    " Amazon Web Services (AWS) ",
    " Google Cloud Platform (GCP) ",
    " Microsoft Azure ",
    " IBM Cloud / Red Hat ",
    " Oracle Cloud ",
    " VMware Cloud ",
    " SAP Cloud ",
    " Tencent Cloud ",
    " Alibaba Cloud ",
    " Huawei Cloud ",
]

# compute and immediately select in the reversed custom order to avoid extra sorts
percentages_cell68 = count_then_return_percent_68(
    responses_df_2022_cell10, question_of_interest_cell68
).loc[responses_in_order_cell68[::-1]]

percentages_cell68.info()

<class 'pandas.core.series.Series'>
Index: 10 entries,  Huawei Cloud  to  Amazon Web Services (AWS) 
Series name: count
Non-Null Count  Dtype  
--------------  -----  
10 non-null     float64
dtypes: float64(1)
memory usage: 160.0+ bytes
CPU times: user 43 ms, sys: 3.84 ms, total: 46.9 ms
Wall time: 42.6 ms


In [62]:
%%time
# accepted
### cell 57 ###


def grab_subset_of_data_69(original_df, question_of_interest):
    # Select only the columns containing the question substring (executes on GPU)
    new_df = original_df.filter(like=question_of_interest)
    # Build the new column names on the CPU (negligible overhead for small list of column names)
    new_cols = [col.split("-", 1)[-1].lstrip() for col in new_df.columns]
    # Assign the cleaned names
    new_df.columns = new_cols
    # Drop rows where all selected columns are null (executes on GPU)
    new_df = new_df.dropna(how="all")
    return new_df


question_of_interest_cell69 = (
    "Do you use any of the following cloud computing products?"
)
compute_df_2022 = grab_subset_of_data_69(
    responses_df_2022_cell10, question_of_interest_cell69
)
compute_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3749 entries, 8 to 23994
Data columns (total 5 columns):
 #   Column                                                  Non-Null Count  Dtype 
---  ------                                                  --------------  ----- 
 0   Selected Choice -  Amazon Elastic Compute Cloud (EC2)   1609 non-null   object
 1   Selected Choice -  Microsoft Azure Virtual Machines     887 non-null    object
 2   Selected Choice -  Google Cloud Compute Engine          1321 non-null   object
 3   Selected Choice - No / None                             930 non-null    object
 4   Selected Choice - Other                                 68 non-null     object
dtypes: object(5)
memory usage: 175.7+ KB
CPU times: user 44.9 ms, sys: 28 μs, total: 44.9 ms
Wall time: 41.9 ms


In [63]:
%%time
### cell 58 ###


def grab_subset_of_data_70(original_df, question_of_interest):
    # Select only the columns that contain the question text (GPU‐side)
    df = original_df.filter(like=question_of_interest)
    # Vectorized split and strip on the Index to get the suffix after “-”
    new_cols = df.columns.str.split("-").str.get(-1).str.lstrip()
    df.columns = new_cols
    # GPU‐side drop of rows where all of these columns are null
    return df.dropna(how="all")


question_of_interest_cell70 = "Do you use any of the following data storage products?"
responses_df_2022_cell10.columns = responses_df_2022_cell10.columns.str.replace(
    "(AWS/GCP/Azure customers only)", "for AWS GCP and Azure customers"
)
storage_df_2022 = grab_subset_of_data_70(
    responses_df_2022_cell10, question_of_interest_cell70
)
storage_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3735 entries, 8 to 23994
Data columns (total 8 columns):
 #   Column                                Non-Null Count  Dtype 
---  ------                                --------------  ----- 
 0   Microsoft Azure Blob Storage          615 non-null    object
 1   Microsoft Azure Files                 511 non-null    object
 2   Amazon Simple Storage Service (S3)    1624 non-null   object
 3   Amazon Elastic File System (EFS)      447 non-null    object
 4   Google Cloud Storage (GCS)            1288 non-null   object
 5   Google Cloud Filestore                481 non-null    object
 6   No / None                             771 non-null    object
 7   Other                                 74 non-null     object
dtypes: object(8)
memory usage: 262.6+ KB
CPU times: user 119 ms, sys: 7.79 ms, total: 127 ms
Wall time: 119 ms


In [64]:
%%time
### cell 59 ###


def grab_subset_of_data_71(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_cell71 = "Do you use any of the following data products (relational databases, data warehouses, data lakes, or similar)?"
big_data_df_2022 = grab_subset_of_data_71(
    responses_df_2022_cell10, question_of_interest_cell71
)
big_data_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4242 entries, 3 to 23994
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   MySQL                          2233 non-null   object 
 1   PostgreSQL                     1516 non-null   object 
 2   SQLite                         1159 non-null   object 
 3   Oracle Database                688 non-null    object 
 4   MongoDB                        1031 non-null   object 
 5   Snowflake                      399 non-null    object 
 6   IBM Db2                        192 non-null    object 
 7   Microsoft SQL Server           1203 non-null   object 
 8   Microsoft Azure SQL Database   520 non-null    object 
 9   Amazon Redshift                380 non-null    object 
 10  Amazon RDS                     505 non-null    object 
 11  Amazon DynamoDB                356 non-null    object 
 12  Google Cloud BigQuery          690 non-null    objec

In [65]:
%%time
# accepted
### cell 60 ###


def grab_subset_of_data_72(original_df, question_of_interest):
    # 1) Use GPU‐accelerated filter to pick matching columns
    df = original_df.filter(like=question_of_interest, axis=1)
    # 2) Rename entirely on the host (metadata only) to avoid a CPU‐bound .rename() call
    df.columns = [col.split("-")[-1].lstrip() for col in df.columns]
    # 3) Drop rows where all values are null via GPU boolean indexing
    mask = df.notnull().any(axis=1)
    return df[mask]


question_of_interest_cell72 = (
    "Do you use any of the following business intelligence tools?"
)
bi_df_2022 = grab_subset_of_data_72(
    responses_df_2022_cell10, question_of_interest_cell72
)
bi_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3127 entries, 13 to 23994
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Amazon QuickSight         224 non-null    object 
 1   Microsoft Power BI        1658 non-null   object 
 2   Google Data Studio        643 non-null    object 
 3   Looker                    166 non-null    object 
 4   Tableau                   1732 non-null   object 
 5   Qlik Sense                207 non-null    object 
 6   Domo                      44 non-null     object 
 7   TIBCO Spotfire            86 non-null     object 
 8   Alteryx                   132 non-null    object 
 9   Sisense                   38 non-null     object 
 10  SAP Analytics Cloud       106 non-null    object 
 11  Microsoft Azure Synapse   167 non-null    object 
 12  Thoughtspot               22 non-null     object 
 13  None                      0 non-null      float64
 14  Other      

In [66]:
%%time
# accepted
### cell 61 ###


def grab_subset_of_data_73(original_df, question_of_interest):
    # GPU-accelerated: select all columns whose names match the question
    subset = original_df.filter(regex=question_of_interest, axis=1)

    # GPU-accelerated: strip everything up through the last '-'
    subset.columns = subset.columns.str.replace(r"^.*-\s*", "", regex=True)

    # GPU-accelerated: drop rows that are all null
    return subset.dropna(how="all")


question_of_interest_cell73 = "Do you use any of the following managed machine learning products on a regular basis?"
managed_ml_df_2022 = grab_subset_of_data_73(
    responses_df_2022_cell10, question_of_interest_cell73
)
managed_ml_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4672 entries, 3 to 23994
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   Amazon SageMaker                561 non-null    object
 1   Azure Machine Learning Studio   396 non-null    object
 2   Google Cloud Vertex AI          350 non-null    object
 3   DataRobot                       123 non-null    object
 4   Databricks                      448 non-null    object
 5   Dataiku                         104 non-null    object
 6   Alteryx                         99 non-null     object
 7   Rapidminer                      113 non-null    object
 8   C3.ai                           30 non-null     object
 9   Domino Data Lab                 52 non-null     object
 10  H2O AI Cloud                    114 non-null    object
 11  No / None                       3016 non-null   object
 12  Other                           121 non-null    obje

In [67]:
%%time
# accepted
### cell 62 ###


def grab_subset_of_data_74(original_df, question_of_interest):
    # select only the columns containing the question text (GPU‐accelerated)
    new_df = (
        original_df.filter(like=question_of_interest)
        # rename by splitting off the prefix up to '-' (runs on GPU)
        .rename(columns=lambda col: col.split("-")[-1].lstrip())
        # drop rows where all selected cols are null (GPU‐accelerated)
        .dropna(how="all")
    )
    return new_df


question_of_interest_cell74 = (
    "Do you use any of the following automated machine learning tools?"
)
automl_df_2022 = grab_subset_of_data_74(
    responses_df_2022_cell10, question_of_interest_cell74
)
automl_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4654 entries, 3 to 23994
Data columns (total 8 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Google Cloud AutoML                463 non-null    object
 1   H2O Driverless AI                  122 non-null    object
 2   Databricks AutoML                  193 non-null    object
 3   DataRobot AutoML                   125 non-null    object
 4   Amazon Sagemaker Autopilot         261 non-null    object
 5   Azure Automated Machine Learning   323 non-null    object
 6   No / None                          3536 non-null   object
 7   Other                              127 non-null    object
dtypes: object(8)
memory usage: 327.2+ KB
CPU times: user 45.7 ms, sys: 3.54 ms, total: 49.2 ms
Wall time: 46.2 ms


In [68]:
%%time
# accepted
### cell 63 ###


def grab_subset_of_data_75(original_df, question_of_interest):
    # Use GPU-backed filter to select columns containing the question text
    sub_df = original_df.filter(like=question_of_interest)
    # Build rename mapping on GPU
    mapping = {col: col.split("-")[-1].lstrip() for col in sub_df.columns}
    # Rename and drop rows that are all null in one go (GPU operations)
    return sub_df.rename(columns=mapping).dropna(how="all")


question_of_interest_cell75 = (
    "Do you use any of the following products to serve your machine learning models?"
)
serving_df_2022 = grab_subset_of_data_75(
    responses_df_2022_cell10, question_of_interest_cell75
)
serving_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1321 entries, 3 to 23989
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   TensorFlow Extended (TFX)   522 non-null    object 
 1   TorchServe                  167 non-null    object 
 2   ONNX Runtime                224 non-null    object 
 3   Triton Inference Server     70 non-null     object 
 4   OpenVINO Model Server       93 non-null     object 
 5   KServe                      63 non-null     object 
 6   BentoML                     53 non-null     object 
 7   Multi Model Server (MMS)    62 non-null     object 
 8   Seldon Core                 47 non-null     object 
 9   MLflow                      621 non-null    object 
 10  None                        0 non-null      float64
 11  Other                       129 non-null    object 
dtypes: float64(1), object(11)
memory usage: 134.2+ KB
CPU times: user 47.3 ms, sys: 4.42 ms, total

In [69]:
%%time
# accepted
### cell 64 ###


def grab_subset_of_data_76(original_df, question_of_interest):
    # GPU‐accelerated column selection
    subset = original_df.filter(like=question_of_interest)
    # Rename by stripping off the question prefix
    renamed = subset.rename(columns=lambda col: col.split("-")[-1].lstrip())
    # Drop rows where all of these new columns are null
    return renamed.dropna(how="all")


question_of_interest_cell76 = "Do you use any tools to help monitor your machine learning models and/or experiments?"

tracking_df_2022 = grab_subset_of_data_76(
    responses_df_2022_cell10, question_of_interest_cell76
)
tracking_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4623 entries, 3 to 23994
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Neptune.ai         128 non-null    object
 1   Weights & Biases   381 non-null    object
 2   Comet.ml           56 non-null     object
 3   TensorBoard        855 non-null    object
 4   Guild.ai           35 non-null     object
 5   ClearML            73 non-null     object
 6   MLflow             659 non-null    object
 7   Aporia             24 non-null     object
 8   Evidently AI       52 non-null     object
 9   Arize              39 non-null     object
 10  WhyLabs            32 non-null     object
 11  Fiddler            65 non-null     object
 12  DVC                143 non-null    object
 13  No / None          2930 non-null   object
 14  Other              124 non-null    object
dtypes: object(15)
memory usage: 577.9+ KB
CPU times: user 52.1 ms, sys: 3.97 ms, total: 56.1 ms
W

In [70]:
%%time
# accepted
### cell 65 ###


def grab_subset_of_data_77(original_df, question_of_interest):
    # select only the columns containing the question text (GPU filter)
    subset = original_df.filter(like=question_of_interest, axis=1)
    # strip off the question prefix from each column name (small Python overhead)
    subset.columns = [col.split("-")[-1].lstrip() for col in subset.columns]
    # drop any rows that are all-null across the remaining columns (GPU dropna)
    return subset.dropna(how="all")


question_of_interest_cell77 = "Do you use any of the following responsible or ethical AI products in your machine learning practices?"
responsible_df_2022 = grab_subset_of_data_77(
    responses_df_2022_cell10, question_of_interest_cell77
)
responsible_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 694 entries, 18 to 23963
Data columns (total 9 columns):
 #   Column                                                                         Non-Null Count  Dtype  
---  ------                                                                         --------------  -----  
 0   Google Responsible AI Toolkit (LIT, What—if, Fairness Indicator, etc)          289 non-null    object 
 1   Microsoft Responsible AI Resources (Fairlearn, Counterfit, InterpretML, etc)   206 non-null    object 
 2   IBM AI Ethics tools (AI Fairness 360, Adversarial Robustness Toolbox, etc      128 non-null    object 
 3   Amazon AI Ethics Tools (Clarify, A2I, etc)                                     184 non-null    object 
 4   The LinkedIn Fairness Toolkit (LiFT)                                           80 non-null     object 
 5   Audit—AI                                                                       69 non-null     object 
 6   Aequitas                     

In [71]:
%%time
### cell 66 ###


def grab_subset_of_data_78(original_df, question_of_interest):
    # Select only the columns containing the question string using GPU-enabled filter
    df = original_df.filter(like=question_of_interest)
    # Build a rename map for the suffixes
    rename_map = {col: col.split("-")[-1].lstrip() for col in df.columns}
    # Apply the rename and drop rows that are all-null (in this subset)
    return df.rename(columns=rename_map).dropna(how="all")


question_of_interest_cell78 = "Do you use any of the following types of specialized hardware when training machine learning models?"
hardware_df_2022 = grab_subset_of_data_78(
    responses_df_2022_cell10, question_of_interest_cell78
)
hardware_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2831 entries, 3 to 23983
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GPUs               2682 non-null   object 
 1   TPUs               653 non-null    object 
 2   IPUs               67 non-null     object 
 3   RDUs               58 non-null     object 
 4   WSEs               26 non-null     object 
 5   Trainium Chips     39 non-null     object 
 6   Inferentia Chips   58 non-null     object 
 7   None               0 non-null      float64
 8   Other              70 non-null     object 
dtypes: float64(1), object(8)
memory usage: 221.2+ KB
CPU times: user 48.6 ms, sys: 193 μs, total: 48.8 ms
Wall time: 45.2 ms


In [72]:
%%time
### cell 67 ###


def grab_subset_of_data_79(original_df, question_of_interest):
    new_df = original_df.copy()
    subset_of_columns = [col for col in new_df.columns if question_of_interest in col]
    mapper = [col.split("-")[-1].lstrip() for col in subset_of_columns]
    mapping_dict = dict(zip(subset_of_columns, mapper))
    new_df = new_df[subset_of_columns].rename(columns=mapping_dict)
    new_df.dropna(how="all", subset=mapper, inplace=True)
    return new_df


question_of_interest_2022 = "Do you use any of the following types of specialized hardware when training machine learning models?"
question_of_interest_2019 = (
    "Which types of specialized hardware do you use on a regular basis?"
)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    question_of_interest_2019, question_of_interest_2022, regex=False
)
question_of_interest_2020 = (
    "Which types of specialized hardware do you use on a regular basis?"
)
responses_df_2020.columns = responses_df_2020.columns.str.replace(
    question_of_interest_2020, question_of_interest_2022, regex=False
)
question_of_interest_2021 = (
    "Which types of specialized hardware do you use on a regular basis?"
)
responses_df_2021.columns = responses_df_2021.columns.str.replace(
    question_of_interest_2021, question_of_interest_2022, regex=False
)
question_of_interest_cell79 = question_of_interest_2022
hardware_df_2019 = grab_subset_of_data_79(
    responses_df_2019_cell10, question_of_interest_cell79
)
counts_2019_cell79 = (
    hardware_df_2019[hardware_df_2019.columns[:]].count().sort_values(ascending=True)
)
percentages_2019_cell79 = counts_2019_cell79 * 100 / len(hardware_df_2019)
percentages_2019_cell79 = percentages_2019_cell79[["TPUs"]]
hardware_df_2020 = grab_subset_of_data_79(
    responses_df_2020, question_of_interest_cell79
)
counts_2020_cell79 = (
    hardware_df_2020[hardware_df_2020.columns[:]].count().sort_values(ascending=True)
)
percentages_2020_cell79 = counts_2020_cell79 * 100 / len(hardware_df_2020)
percentages_2020_cell79 = percentages_2020_cell79[["TPUs"]]
hardware_df_2021 = grab_subset_of_data_79(
    responses_df_2021, question_of_interest_cell79
)
counts_2021_cell79 = (
    hardware_df_2021[hardware_df_2021.columns[:]].count().sort_values(ascending=True)
)
percentages_2021_cell79 = counts_2021_cell79 * 100 / len(hardware_df_2021)
percentages_2021_cell79 = percentages_2021_cell79.rename(
    {"Google Cloud TPUs ": "TPUs", "NVIDIA GPUs ": "GPUs"}
)
percentages_2021_cell79 = percentages_2021_cell79[["TPUs"]]
hardware_df_2022_cell79 = grab_subset_of_data_79(
    responses_df_2022_cell10, question_of_interest_cell79
)
counts_2022_cell79 = (
    hardware_df_2022_cell79[hardware_df_2022_cell79.columns[:]]
    .count()
    .sort_values(ascending=True)
)
percentages_2022_cell79 = counts_2022_cell79 * 100 / len(hardware_df_2022_cell79)
percentages_2022_cell79 = percentages_2022_cell79.rename(
    {"TPUs ": "TPUs", "GPUs ": "GPUs"}
)
percentages_2022_cell79 = percentages_2022_cell79[["TPUs"]]
(
    percentages_2019_cell79.info(),
    percentages_2020_cell79.info(),
    percentages_2021_cell79.info(),
    percentages_2022_cell79.info(),
)

<class 'pandas.core.series.Series'>
Index: 1 entries, TPUs to TPUs
Series name: None
Non-Null Count  Dtype  
--------------  -----  
1 non-null      float64
dtypes: float64(1)
memory usage: 16.0+ bytes
<class 'pandas.core.series.Series'>
Index: 1 entries, TPUs to TPUs
Series name: None
Non-Null Count  Dtype  
--------------  -----  
1 non-null      float64
dtypes: float64(1)
memory usage: 16.0+ bytes
<class 'pandas.core.series.Series'>
Index: 1 entries, TPUs to TPUs
Series name: None
Non-Null Count  Dtype  
--------------  -----  
1 non-null      float64
dtypes: float64(1)
memory usage: 16.0+ bytes
<class 'pandas.core.series.Series'>
Index: 1 entries, TPUs to TPUs
Series name: None
Non-Null Count  Dtype  
--------------  -----  
1 non-null      float64
dtypes: float64(1)
memory usage: 16.0+ bytes
CPU times: user 851 ms, sys: 155 ms, total: 1.01 s
Wall time: 964 ms


(None, None, None, None)

In [73]:
%%time
# accepted
### cell 68 ###

question_of_interest_cell80 = (
    "Approximately how many times have you used a TPU (tensor processing unit)?"
)
responses_in_order_cell80 = [
    "Never",
    "Once",
    "2-5 times",
    "6-25 times",
    "More than 25 times",
]

# compute percentages directly on GPU using normalize in value_counts
percentages_cell80 = (
    responses_df_2022_cell10[question_of_interest_cell80]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(1)
    .sort_index()
    .loc[responses_in_order_cell80]
)

percentages_cell80.info()

<class 'pandas.core.series.Series'>
Index: 5 entries, Never to More than 25 times
Series name: proportion
Non-Null Count  Dtype  
--------------  -----  
5 non-null      float64
dtypes: float64(1)
memory usage: 80.0+ bytes
CPU times: user 41.4 ms, sys: 4.16 ms, total: 45.5 ms
Wall time: 41.6 ms


In [74]:
%%time
### cell 69 ###


def count_then_return_percent_81(dataframe, x_axis_title):
    counts = dataframe[x_axis_title].value_counts(dropna=False)
    percentages = round(counts * 100 / dataframe[x_axis_title].count(), 1)
    return percentages


question_of_interest_2022_cell81 = (
    "Approximately how many times have you used a TPU (tensor processing unit)?"
)
question_of_interest_2019_cell81 = "Have you ever used a TPU (tensor processing unit)?"
responses_df_2019_cell10[question_of_interest_2019_cell81] = responses_df_2019_cell10[
    question_of_interest_2019_cell81
].str.replace("6-24 times", "6-25 times", regex=False)
responses_df_2019_cell10[question_of_interest_2019_cell81] = responses_df_2019_cell10[
    question_of_interest_2019_cell81
].str.replace("> 25 times", "More than 25 times", regex=False)
responses_df_2019_cell10.columns = responses_df_2019_cell10.columns.str.replace(
    question_of_interest_2019_cell81, question_of_interest_2022_cell81, regex=False
)
question_of_interest_cell81 = question_of_interest_2022_cell81
df_2022 = pd.DataFrame(
    count_then_return_percent_81(
        responses_df_2022_cell10, question_of_interest_cell81
    ).sort_index()
)
df_2021 = pd.DataFrame(
    count_then_return_percent_81(
        responses_df_2021, question_of_interest_cell81
    ).sort_index()
)
df_2020 = pd.DataFrame(
    count_then_return_percent_81(
        responses_df_2020, question_of_interest_cell81
    ).sort_index()
)
df_2019 = pd.DataFrame(
    count_then_return_percent_81(
        responses_df_2019_cell10, question_of_interest_cell81
    ).sort_index()
)
df_2022["year"] = "2022"
df_2021["year"] = "2021"
df_2020["year"] = "2020"
df_2019["year"] = "2019"
df_combined = pd.concat([df_2019, df_2020, df_2021, df_2022])
df_combined[x_axis_title] = df_combined.index
x_axis_title_cell81 = "Frequency of TPU usage"
df_combined.columns = ["percentage", "year", x_axis_title_cell81]
question_of_interest_cell81 = question_of_interest_2022_cell81
df_combined_cell81 = df_combined.sort_values(by=["year", "percentage"], ascending=True)
df_combined_cell81.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24 entries, More than 25 times to nan
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   percentage              24 non-null     float64
 1   year                    24 non-null     object 
 2   Frequency of TPU usage  20 non-null     object 
dtypes: float64(1), object(2)
memory usage: 768.0+ bytes
CPU times: user 397 ms, sys: 21.3 ms, total: 419 ms
Wall time: 395 ms


In [75]:
%%time
# accepted
### cell 70 ###


def grab_subset_of_data_82(original_df, question_of_interest):
    # Select only columns containing the question (GPU filter)
    subset_df = original_df.filter(like=question_of_interest, axis=1)
    # Build a mapping from full column names to the trimmed suffix
    mapping_dict = {col: col.split("-")[-1].lstrip() for col in subset_df.columns}
    # Drop rows where all of these columns are null (GPU) and then rename (GPU)
    return subset_df.dropna(how="all").rename(columns=mapping_dict)


question_of_interest_cell82 = (
    "Who/what are your favorite media sources that report on data science topics?"
)
media_df_2022 = grab_subset_of_data_82(
    responses_df_2022_cell10, question_of_interest_cell82
)
media_df_2022.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18423 entries, 2 to 23996
Data columns (total 12 columns):
 #   Column                                                                      Non-Null Count  Dtype  
---  ------                                                                      --------------  -----  
 0   Twitter (data science influencers)                                          3995 non-null   object 
 1   Email newsletters (Data Elixir, O'Reilly Data & AI, etc)                    3787 non-null   object 
 2   Reddit (r/machinelearning, etc)                                             2678 non-null   object 
 3   Kaggle (notebooks, forums, etc)                                             11181 non-null  object 
 4   Course Forums (forums.fast.ai, Coursera forums, etc)                        4006 non-null   object 
 5   YouTube (Kaggle YouTube, Cloud AI Adventures, etc)                          11957 non-null  object 
 6   Podcasts (Chai Time Data Science, OâReilly Data S

In [76]:
%%time
### cell 71 ###

(
    responses_df_2017.shape[0],
    responses_df_2018_cell10.shape[0],
    responses_df_2019_cell10.shape[0],
    responses_df_2020.shape[0],
    responses_df_2021.shape[0],
    responses_df_2022_cell10.shape[0],
)

CPU times: user 138 ms, sys: 4.23 ms, total: 142 ms
Wall time: 134 ms


(16716, 23859, 19717, 20036, 25973, 23997)

In [77]:
%%time
### cell 72 ###

responses_only_duration = responses_df_2022_cell10["Duration (in seconds)"]
responses_only_duration_cell86 = pd.DataFrame(
    pd.to_numeric(responses_only_duration, errors="coerce") / 60
)
responses_only_duration_cell86.columns = ["Duration (in minutes)"]
median = round(responses_only_duration_cell86["Duration (in minutes)"].median(), 0)
print("The median response time was approximately", median, "minutes.")
responses_only_duration_longer_5m = responses_only_duration_cell86[
    responses_only_duration_cell86["Duration (in minutes)"] > 5
]
print(
    "The total number of respondents that took more than 5 minutes was",
    responses_only_duration_longer_5m.shape[0],
    "out of",
    responses_df_2022_cell10.shape[0],
)

The median response time was approximately 7.0 minutes.
The total number of respondents that took more than 5 minutes was 16324 out of 23997
CPU times: user 81.3 ms, sys: 4.53 ms, total: 85.8 ms
Wall time: 77.9 ms


In [78]:
end = time.time()
print(end - start)

33.61940622329712


**Future directions to consider:**
* Divide the population into interesting subgroups and identify interesting insights.

 * Do students have different preferences as compared to professionals?
 * Do GCP customers have different preferences as compared to AWS customers?
 * Which cloud computing platforms have seen the most growth in recent years?
 * Do salaries scale according to experience levels?  What traits might predict having a very high salary?



**Credits / Attribution:**

* The idea to use pattern matching to identify which columns are associated with which questions came from @siddhantsadangi's notebook ["Your Country VS The World"](https://www.kaggle.com/siddhantsadangi/your-country-vs-the-world-24-factors-wip) (see function grab_subset_of_data() for more detail).
* Most plotting functions were adapted from examples in the [plotly documentation](https://plotly.com/python/plotly-express/).
* This notebook (and every other public notebook on Kaggle) was released under an [Apache 2.0 license](https://www.apache.org/licenses/LICENSE-2.0). Consider clicking on the "copy & edit" button in the top right corner and then you can focus on some subset of the community that you find to be interesting!